# Cell 00. 제목·분석 계약

# 2024학년도 정시 입시결과 크롤 데이터 EDA

- 프로젝트명: 2024학년도 정시(수능위주전형) 입시결과 크롤 데이터 품질검증
- 분석일: 실행 시각 기준 자동 기록
- 입시 결과연도: 2024학년도
- 크롤 페이지 조회연도: 2025
- 분석 대상 대학 수: 시드 52개, 고유 크롤 대상 기준 51개
- 입력 파일: `00_crawl_seed_university_2024.csv`, `01_crawl_source_registry.csv`, `02_admission_result_raw_2024.parquet`/`.csv`, `raw_html/`
- 출력 폴더: `workbook/p2/p2_2/data/crawl_2024_admission/eda_outputs`
- 현재 단계: CRISP-DM Data Understanding, Data Preparation Readiness, Modeling Readiness, Evaluation, Deployment
- 범위 외 작업: 웹 재수집, H1/H2 통계모형 적합, 자동 fuzzy merge, 학점자료 최종 병합

## Business Understanding

2024학년도 대학-모집단위별 입결 데이터를 2025년 대학-학과별 A학점 비율과 결합하기 전에,
크롤링 산출물이 다음 단계의 분석 입력으로 사용할 수 있는지 검증한다.

향후 분석 흐름:

```text
2024학년도 대학-모집단위별 입결
→ 2025년 대학-학과별 A학점 비율과 결합
→ H1: 입결 수준과 A학점 비율의 관련성 검정
→ H2: 학점포기제와 A학점 비율의 관련성 검정
```

이번 노트북의 핵심은 크롤 성공률 확인을 넘어 대학별 입결 필드 가용성, 비교 가능한 점수의 범위,
파서 오류, 중복 구조, 향후 모집단위-학점학과 매핑 난이도를 정량화하는 것이다.

Target 설계: 현재 데이터에는 최종 타깃 `a_rate_pct`가 없으므로 모델 타깃을 만들지 않고, 향후 H1 피처 후보의 품질과 비교가능성만 평가한다.  
관찰: `percentile_70cut_num`은 대학 간 비교 가능성이 가장 높은 후보 입결 지표다.  
원인 가설: 대학별 환산점수는 대학 내부 산식의 산물이라 직접 비교하면 왜곡될 수 있다.  
제한: 본 노트북은 adiga 캐시와 크롤 산출물만 사용하며 입학처 원문 교차검증은 표본 수준으로 제한한다.  
결론: Gate 2에서는 모집요강 반영규칙 수집과 모집단위 crosswalk가 우선이다.

In [1]:
# Cell 01. 설정 및 라이브러리
from pathlib import Path
import re
import json
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    SCIPY_IMPORT_ERROR = repr(exc)

try:
    from bs4 import BeautifulSoup
    BS4_AVAILABLE = True
except Exception as exc:
    BS4_AVAILABLE = False
    BS4_IMPORT_ERROR = repr(exc)

PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
NOTEBOOK_PATH = PROJECT_ROOT / "workbook/p2/p2_2/crawl_eda.ipynb"
NOTEBOOK_DIR = NOTEBOOK_PATH.parent
BASE_DIR = PROJECT_ROOT / "workbook/p2/p2_2/data/crawl_2024_admission"
OUTPUT_DIR = BASE_DIR / "eda_outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED_PATH = BASE_DIR / "00_crawl_seed_university_2024.csv"
REGISTRY_PATH = BASE_DIR / "01_crawl_source_registry.csv"
RAW_PARQUET_PATH = BASE_DIR / "02_admission_result_raw_2024.parquet"
RAW_CSV_PATH = BASE_DIR / "02_admission_result_raw_2024.csv"
RAW_HTML_DIR = BASE_DIR / "raw_html"

RANDOM_STATE = 20260710
EXPECTED_SEED_N = 52
EXPECTED_UNIQUE_CRAWL_N = 51
EXPECTED_RAW_ROWS = 3096
EXPECTED_HTML_N = 51
TOLERANCE = 0
RUN_STARTED_AT = datetime.now()

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 170,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

MISSING_TOKENS = {
    "",
    "-",
    "–",
    "—",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "미공개",
    "없음",
    "해당없음",
    "※",
    "ㆍ",
}
CSV_ENCODINGS = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
AUDIT_LOG = []

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")

def audit(step, dataset, action, rows_before=None, rows_after=None, rows_affected=None, status="INFO", rule="", note=""):
    AUDIT_LOG.append(
        {
            "step": step,
            "dataset": dataset,
            "action": action,
            "rows_before": rows_before,
            "rows_after": rows_after,
            "rows_affected": rows_affected,
            "status": status,
            "rule": rule,
            "note": note,
            "executed_at": now_iso(),
        }
    )

def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index, encoding="utf-8-sig")
    audit(
        "artifact_export",
        path.name,
        "save_csv",
        rows_before=len(df),
        rows_after=len(df),
        rows_affected=len(df),
        status="PASS",
        rule="utf-8-sig",
        note=str(path),
    )
    return path

def save_figure(fig, file_name: str) -> Path:
    path = FIGURE_DIR / file_name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    audit(
        "artifact_export",
        file_name,
        "save_png",
        status="PASS",
        note=str(path),
    )
    return path

def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def nonempty_mask(series: pd.Series) -> pd.Series:
    normalized = series.map(normalize_text).str.lower()
    return ~normalized.isin(MISSING_TOKENS)

def shorten_label(value, max_len: int = 22) -> str:
    text = normalize_text(value)
    return text if len(text) <= max_len else text[: max_len - 1] + "…"

def status_against_expected(actual, expected, warn_note=""):
    if actual == expected:
        return "EXPECTED"
    if warn_note:
        return "WARN"
    return "FAIL"

display(
    pd.DataFrame(
        [
            {"key": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
            {"key": "BASE_DIR", "value": str(BASE_DIR)},
            {"key": "OUTPUT_DIR", "value": str(OUTPUT_DIR)},
            {"key": "RUN_STARTED_AT", "value": RUN_STARTED_AT.isoformat(timespec="seconds")},
            {"key": "SCIPY_AVAILABLE", "value": SCIPY_AVAILABLE},
            {"key": "BS4_AVAILABLE", "value": BS4_AVAILABLE},
        ]
    )
)
audit("setup", "environment", "initialize_paths_and_imports", status="PASS")

,key,value
0,PROJECT_ROOT,/home/sieg/projects-wsl/SBS_dataScience
1,BASE_DIR,/home/sieg/projects-wsl/SBS_dataScience/workbo...
2,OUTPUT_DIR,/home/sieg/projects-wsl/SBS_dataScience/workbo...
3,RUN_STARTED_AT,2026-07-14T14:21:02
4,SCIPY_AVAILABLE,True
5,BS4_AVAILABLE,True


In [2]:
# Cell 02. 파일 인벤토리
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    if not path.exists() or not path.is_file():
        return ""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def file_row(label: str, path: Path) -> dict:
    exists = path.exists()
    stat = path.stat() if exists else None
    return {
        "file_name": label,
        "absolute_path": str(path),
        "exists": exists,
        "file_size_bytes": stat.st_size if stat and path.is_file() else np.nan,
        "modified_at": datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds") if stat else "",
        "sha256": sha256_file(path) if exists and path.is_file() else "",
        "html_file_count": np.nan,
        "html_total_size": np.nan,
        "html_zero_byte_count": np.nan,
        "duplicate_sha256_count": np.nan,
    }

inventory_rows = [
    file_row("00_crawl_seed_university_2024.csv", SEED_PATH),
    file_row("01_crawl_source_registry.csv", REGISTRY_PATH),
    file_row("02_admission_result_raw_2024.parquet", RAW_PARQUET_PATH),
    file_row("02_admission_result_raw_2024.csv", RAW_CSV_PATH),
]

html_files = sorted(RAW_HTML_DIR.glob("*.html")) if RAW_HTML_DIR.exists() else []
html_hashes = [sha256_file(p) for p in html_files]
html_hash_counts = pd.Series(html_hashes, dtype="string").value_counts(dropna=True)
duplicate_html_sha256_count = int((html_hash_counts[html_hash_counts > 1] - 1).sum()) if len(html_hash_counts) else 0
html_total_size = int(sum(p.stat().st_size for p in html_files)) if html_files else 0
html_zero_byte_count = int(sum(p.stat().st_size == 0 for p in html_files)) if html_files else 0
inventory_rows.append(
    {
        "file_name": "raw_html/",
        "absolute_path": str(RAW_HTML_DIR),
        "exists": RAW_HTML_DIR.exists(),
        "file_size_bytes": np.nan,
        "modified_at": datetime.fromtimestamp(RAW_HTML_DIR.stat().st_mtime).isoformat(timespec="seconds") if RAW_HTML_DIR.exists() else "",
        "sha256": "",
        "html_file_count": len(html_files),
        "html_total_size": html_total_size,
        "html_zero_byte_count": html_zero_byte_count,
        "duplicate_sha256_count": duplicate_html_sha256_count,
    }
)

file_inventory = pd.DataFrame(inventory_rows)
save_csv(file_inventory, OUTPUT_DIR / "eda_01_file_inventory.csv")
audit("file_inventory", "input_files", "profile_files_and_html_cache", rows_after=len(file_inventory), status="PASS")
display(file_inventory)

,file_name,absolute_path,exists,file_size_bytes,modified_at,sha256,html_file_count,html_total_size,html_zero_byte_count,duplicate_sha256_count
0,00_crawl_seed_university_2024.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,6428.0,2026-07-10T13:00:25,e9b26b12a45b4639144d9fd26d9c477d747d6dc3fc6327...,NaN,NaN,NaN,NaN
1,01_crawl_source_registry.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,22492.0,2026-07-10T13:00:35,71cb64b6d65b572ffaa2d863a90df8f16f3acb214a3da3...,NaN,NaN,NaN,NaN
2,02_admission_result_raw_2024.parquet,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,203093.0,2026-07-10T13:00:35,c0c005379abea4cd6fb101969f6c2e76d6e51e6b437468...,NaN,NaN,NaN,NaN
3,02_admission_result_raw_2024.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1578623.0,2026-07-10T13:00:35,667be6581b71d685dab21b0952855b0b6c4dbc73d162a8...,NaN,NaN,NaN,NaN
4,raw_html/,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,NaN,2026-07-10T12:03:41,,51.0,89473850.0,0.0,0.0


In [3]:
# Cell 03. 안전한 데이터 로딩
def read_csv_safe(path: Path):
    errors = []
    for encoding in CSV_ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=encoding)
            return df, encoding, ""
        except Exception as exc:
            errors.append(f"{encoding}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"CSV 로딩 실패: {path}\n" + "\n".join(errors))

seed_raw, seed_encoding, _ = read_csv_safe(SEED_PATH)
registry_raw, registry_encoding, _ = read_csv_safe(REGISTRY_PATH)
raw_csv_df, raw_csv_encoding, _ = read_csv_safe(RAW_CSV_PATH) if RAW_CSV_PATH.exists() else (None, "", "missing")

raw_parquet_df = None
parquet_error = ""
if RAW_PARQUET_PATH.exists():
    try:
        raw_parquet_df = pd.read_parquet(RAW_PARQUET_PATH)
    except Exception as exc:
        parquet_error = repr(exc)

def normalized_string_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.astype("string").fillna("").replace("<NA>", "")

compare_rows = []
if raw_parquet_df is not None and raw_csv_df is not None:
    parquet_norm = normalized_string_df(raw_parquet_df)
    csv_norm = normalized_string_df(raw_csv_df)
    key_col = "raw_row_id" if "raw_row_id" in raw_parquet_df.columns and "raw_row_id" in raw_csv_df.columns else None
    same_key_set = bool(set(parquet_norm[key_col]) == set(csv_norm[key_col])) if key_col else np.nan
    same_rows_after_sort = False
    if key_col and same_key_set:
        left = parquet_norm.sort_values(key_col).reset_index(drop=True)
        right = csv_norm.sort_values(key_col).reset_index(drop=True)
        same_rows_after_sort = bool(left.equals(right))
    compare_rows.append(
        {
            "comparison": "parquet_vs_csv",
            "parquet_shape": str(raw_parquet_df.shape),
            "csv_shape": str(raw_csv_df.shape),
            "same_shape": raw_parquet_df.shape == raw_csv_df.shape,
            "same_columns": list(raw_parquet_df.columns) == list(raw_csv_df.columns),
            "key_column": key_col or "",
            "same_key_set": same_key_set,
            "same_rows_after_sort_normalized": same_rows_after_sort,
            "note": "정규화 비교는 NaN/빈 문자열 차이를 완화해 비교함",
        }
    )

if raw_parquet_df is not None:
    admission_raw = raw_parquet_df.copy()
    admission_load_source = "parquet"
elif raw_csv_df is not None:
    admission_raw = raw_csv_df.copy()
    admission_load_source = "csv"
else:
    raise FileNotFoundError("Parquet과 CSV 원본 결과가 모두 없습니다.")

for name, df in [("seed_raw", seed_raw), ("registry_raw", registry_raw), ("admission_raw", admission_raw)]:
    if "source_row_id" not in df.columns:
        df.insert(0, "source_row_id", np.arange(len(df)))
    audit("safe_loading", name, "load_dataframe", rows_after=len(df), status="PASS", note=f"shape={df.shape}")

load_report = pd.DataFrame(
    [
        {"dataset": "seed_raw", "shape": str(seed_raw.shape), "encoding_or_source": seed_encoding},
        {"dataset": "registry_raw", "shape": str(registry_raw.shape), "encoding_or_source": registry_encoding},
        {"dataset": "admission_raw", "shape": str(admission_raw.shape), "encoding_or_source": admission_load_source},
        {"dataset": "raw_csv", "shape": str(raw_csv_df.shape) if raw_csv_df is not None else "", "encoding_or_source": raw_csv_encoding},
        {"dataset": "raw_parquet", "shape": str(raw_parquet_df.shape) if raw_parquet_df is not None else "", "encoding_or_source": parquet_error or "read_ok"},
    ]
)
parquet_csv_compare = pd.DataFrame(compare_rows)
display(load_report)
if len(parquet_csv_compare):
    display(parquet_csv_compare)

,dataset,shape,encoding_or_source
0,seed_raw,"(52, 13)",utf-8-sig
1,registry_raw,"(51, 19)",utf-8-sig
2,admission_raw,"(3096, 19)",parquet
3,raw_csv,"(3096, 18)",utf-8-sig
4,raw_parquet,"(3096, 18)",read_ok


,comparison,parquet_shape,csv_shape,same_shape,same_columns,key_column,same_key_set,same_rows_after_sort_normalized,note
0,parquet_vs_csv,"(3096, 18)","(3096, 18)",True,True,raw_row_id,True,True,정규화 비교는 NaN/빈 문자열 차이를 완화해 비교함


In [4]:
# Cell 04. 기본 구조 프로파일링
def numeric_conversion_rate(series: pd.Series) -> float:
    mask = nonempty_mask(series)
    if int(mask.sum()) == 0:
        return np.nan
    cleaned = (
        series.map(normalize_text)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace(":1", "", regex=False)
        .str.replace("：1", "", regex=False)
    )
    cleaned = cleaned.str.extract(r"(-?\d+(?:\.\d+)?)", expand=False)
    converted = pd.to_numeric(cleaned, errors="coerce")
    return float(converted[mask].notna().mean())

def sample_values(series: pd.Series, n: int = 5) -> str:
    vals = [normalize_text(v) for v in series.dropna().unique()[:n]]
    return json.dumps(vals, ensure_ascii=False)

schema_rows = []
missing_rows = []
dataset_map = {"seed_raw": seed_raw, "registry_raw": registry_raw, "admission_raw": admission_raw}
for dataset, df in dataset_map.items():
    display(Markdown(f"### {dataset}: shape={df.shape}"))
    display(df.head(5))
    display(df.tail(5))
    display(df.sample(min(5, len(df)), random_state=RANDOM_STATE))
    duplicate_n = int(df.duplicated().sum())
    memory_bytes = int(df.memory_usage(deep=True).sum())
    audit(
        "schema_profile",
        dataset,
        "profile_dataframe",
        rows_after=len(df),
        rows_affected=duplicate_n,
        status="PASS",
        note=f"duplicate_n={duplicate_n}, memory_bytes={memory_bytes}",
    )
    for col in df.columns:
        null_n = int(df[col].isna().sum())
        non_null_n = int(df[col].notna().sum())
        schema_rows.append(
            {
                "dataset": dataset,
                "column": col,
                "dtype": str(df[col].dtype),
                "non_null_n": non_null_n,
                "null_n": null_n,
                "null_rate": null_n / len(df) if len(df) else np.nan,
                "unique_n": int(df[col].nunique(dropna=True)),
                "sample_values": sample_values(df[col]),
                "numeric_conversion_rate": numeric_conversion_rate(df[col]),
                "memory_bytes_dataset": memory_bytes,
                "complete_duplicate_rows_dataset": duplicate_n,
            }
        )
        missing_rows.append(
            {
                "dataset": dataset,
                "column": col,
                "null_n": null_n,
                "null_rate": null_n / len(df) if len(df) else np.nan,
            }
        )

schema_profile = pd.DataFrame(schema_rows)
missingness_profile = pd.DataFrame(missing_rows)
save_csv(schema_profile, OUTPUT_DIR / "eda_02_schema_profile.csv")
save_csv(missingness_profile, OUTPUT_DIR / "eda_03_missingness_profile.csv")
display(schema_profile.head(20))

### seed_raw: shape=(52, 13)

,source_row_id,univ_id,univ_name_raw,univ_name_std,campus_id,campus_name_std,adiga_univ_code,adiga_label,target_institution_flag,branch_status,crawl_priority,manual_review_required,seed_note
0,0,U0000019,서울대,서울대학교,0000019,본교/미구분,19,서울대학교[본교],True,main,1,False,NaN
1,1,U0000149,연세대,연세대학교,0000149,본교/미구분,149,연세대학교[본교],True,main,2,False,NaN
2,2,U0000069,고려대,고려대학교,0000069,본교/미구분,69,고려대학교[본교],True,main,3,False,NaN
3,3,U0000133,성균관대,성균관대학교,0000133,본교/미구분,133,성균관대학교[본교],True,main,4,False,NaN
4,4,U0000203,한양대,한양대학교,0000203,본교/미구분,203,한양대학교[본교],True,main,5,False,NaN


,source_row_id,univ_id,univ_name_raw,univ_name_std,campus_id,campus_name_std,adiga_univ_code,adiga_label,target_institution_flag,branch_status,crawl_priority,manual_review_required,seed_note
47,47,U0000003,강원대,강원대학교,0000003,본교/미구분,3,강원대학교[본교],True,main,48,False,NaN
48,48,U0000150,연세대(원주캠),연세대학교,0000150,본교/미구분,150,연세대학교(미래)[분교],True,branch,49,False,NaN
49,49,U0000070,고려대(세종캠),고려대학교,0000070,본교/미구분,70,고려대학교(세종)[분교],True,branch,50,False,NaN
50,50,U0000140,수원대,수원대학교,0000140,본교/미구분,140,수원대학교[본교],True,main,51,False,NaN
51,51,U0000247,한국공대,한국공학대학교,0000247,본교/미구분,247,한국공학대학교[본교],True,main,52,False,NaN


,source_row_id,univ_id,univ_name_raw,univ_name_std,campus_id,campus_name_std,adiga_univ_code,adiga_label,target_institution_flag,branch_status,crawl_priority,manual_review_required,seed_note
38,38,U0000126,서울여대,서울여자대학교,0000126,본교/미구분,126,서울여자대학교[본교],True,main,39,False,NaN
15,15,U0000141,숙명여대,숙명여자대학교,0000141,본교/미구분,141,숙명여자대학교[본교],True,main,16,False,NaN
49,49,U0000070,고려대(세종캠),고려대학교,0000070,본교/미구분,70,고려대학교(세종)[분교],True,branch,50,False,NaN
4,4,U0000203,한양대,한양대학교,0000203,본교/미구분,203,한양대학교[본교],True,main,5,False,NaN
51,51,U0000247,한국공대,한국공학대학교,0000247,본교/미구분,247,한국공학대학교[본교],True,main,52,False,NaN


### registry_raw: shape=(51, 19)

...

### admission_raw: shape=(3096, 19)

,source_row_id,raw_row_id,source_id,univ_id,admission_year,raw_table_index,raw_row_index,raw_section_title,raw_header_json,raw_cells_json,raw_admission_group,raw_recruitment_unit,raw_recruitment_n,raw_competition_rate,raw_additional_rank,raw_score_70cut,raw_score_max,raw_percentile_70cut,raw_parse_warning
0,0,SRC_0000019_2025_0001_T71_R0,SRC_0000019_2025_0001,U0000019,2024,71,0,정시 지역균형전형,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""‘나’군"", ""인문계열"", ""23"", ""2.48:1"", ""2"", ""401.9"",...",‘나’군,인문계열,23,2.48:1,2,401.9,-,96.25,
1,1,SRC_0000019_2025_0001_T71_R1,SRC_0000019_2025_0001,U0000019,2024,71,1,정시 지역균형전형,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""‘나’군"", ""정치외교학부"", ""10"", ""3.10:1"", ""0"", ""405.7...",‘나’군,정치외교학부,10,3.10:1,0,405.7,-,98.25,1개 선두 컬럼 rowspan 승계
2,2,SRC_0000019_2025_0001_T71_R2,SRC_0000019_2025_0001,U0000019,2024,71,2,정시 지역균형전형,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""‘나’군"", ""경제학부"", ""20"", ""1.95:1"", ""3"", ""404.5"",...",‘나’군,경제학부,20,1.95:1,3,404.5,-,98,1개 선두 컬럼 rowspan 승계
3,3,SRC_0000019_2025_0001_T71_R3,SRC_0000019_2025_0001,U0000019,2024,71,3,정시 지역균형전형,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""‘나’군"", ""인류학과"", ""7"", ""4.57:1"", ""0"", ""401.1"", ...",‘나’군,인류학과,7,4.57:1,0,401.1,-,97.25,1개 선두 컬럼 rowspan 승계
4,4,SRC_0000019_2025_0001_T71_R4,SRC_0000019_2025_0001,U0000019,2024,71,4,정시 지역균형전형,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""‘나’군"", ""광역"", ""46"", ""2.87:1"", ""2"", ""414.1"", ""...",‘나’군,광역,46,2.87:1,2,414.1,-,96.75,1개 선두 컬럼 rowspan 승계


,source_row_id,raw_row_id,source_id,univ_id,admission_year,raw_table_index,raw_row_index,raw_section_title,raw_header_json,raw_cells_json,raw_admission_group,raw_recruitment_unit,raw_recruitment_n,raw_competition_rate,raw_additional_rank,raw_score_70cut,raw_score_max,raw_percentile_70cut,raw_parse_warning
3091,3091,SRC_0000247_2025_0051_T7_R15,SRC_0000247_2025_0051,U0000247,2024,7,15,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""다군"", ""메카트로닉스공학부 ( 메카트로닉스전공 )"", ""1"", ""4"", ""1""...",다군,메카트로닉스공학부 ( 메카트로닉스전공 ),1,4,1,241.56,400,57.63,1개 선두 컬럼 rowspan 승계
3092,3092,SRC_0000247_2025_0051_T7_R16,SRC_0000247_2025_0051,U0000247,2024,7,16,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""다군"", ""전자공학부 ( 임베디드시스템전공 )"", ""1"", ""10"", ""1"", ...",다군,전자공학부 ( 임베디드시스템전공 ),1,10,1,-,400,-,1개 선두 컬럼 rowspan 승계
3093,3093,SRC_0000247_2025_0051_T7_R17,SRC_0000247_2025_0051,U0000247,2024,7,17,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""다군"", ""나노반도체공학과"", ""1"", ""5"", ""1"", ""300.88"", ""4...",다군,나노반도체공학과,1,5,1,300.88,400,71.38,1개 선두 컬럼 rowspan 승계
3094,3094,SRC_0000247_2025_0051_T7_R18,SRC_0000247_2025_0051,U0000247,2024,7,18,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""다군"", ""경영학부 ( 경영학전공 )"", ""1"", ""7"", ""0"", ""303.2...",다군,경영학부 ( 경영학전공 ),1,7,0,303.2,400,73.0,1개 선두 컬럼 rowspan 승계
3095,3095,SRC_0000247_2025_0051_T7_R19,SRC_0000247_2025_0051,U0000247,2024,7,19,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""다군"", ""디자인공학부 ( 산업디자인공학전공 )"", ""1"", ""6"", ""3"", ...",다군,디자인공학부 ( 산업디자인공학전공 ),1,6,3,247.2,400,61.0,1개 선두 컬럼 rowspan 승계


,source_row_id,raw_row_id,source_id,univ_id,admission_year,raw_table_index,raw_row_index,raw_section_title,raw_header_json,raw_cells_json,raw_admission_group,raw_recruitment_unit,raw_recruitment_n,raw_competition_rate,raw_additional_rank,raw_score_70cut,raw_score_max,raw_percentile_70cut,raw_parse_warning
2233,2233,SRC_0000056_2025_0037_T8_R29,SRC_0000056_2025_0037,U0000056,2024,8,29,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""나군"", ""스마트시티공학부"", ""26"", ""5.54"", ""32"", ""81.365...",나군,스마트시티공학부,26,5.54,32,81.365,100,79.750,1개 선두 컬럼 rowspan 승계
2674,2674,SRC_0000013_2025_0044_T8_R39,SRC_0000013_2025_0044,U0000013,2024,8,39,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""나군"", ""에너지수송시스템공학부 ( 냉동공조공학전공 , 기계시스템공학전공 , 조...",나군,"에너지수송시스템공학부 ( 냉동공조공학전공 , 기계시스템공학전공 , 조선해양시스템공학...",27,3,8,675,"1,000",73,1개 선두 컬럼 rowspan 승계
1162,1162,SRC_0000078_2025_0017_T7_R34,SRC_0000078_2025_0017,U0000078,2024,7,34,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""나군"", ""정치외교학과"", ""18"", ""5.67"", ""41"", ""682.3"", ...",나군,정치외교학과,18,5.67,41,682.3,"1,000",86.75,
617,617,SRC_0000175_2025_0007_T7_R5,SRC_0000175_2025_0007,U0000175,2024,7,5,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 순위"", ""대학...","[""가군"", ""미디어커뮤니케이션학부"", ""30"", ""3.6"", ""8"", ""755.8...",가군,미디어커뮤니케이션학부,30,3.6,8,755.89,"1,000",90.90,1개 선두 컬럼 rowspan 승계
1449,1449,SRC_0000005_2025_0022_T7_R41,SRC_0000005_2025_0022,U0000005,2024,7,41,NaN,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 최종번호"", ""...","[""나군"", ""식물의학과"", ""7"", ""7.9 :1"", ""12"", ""645.38"",...",나군,식물의학과,7,7.9 :1,12,645.38,"1,000",238.00,


,dataset,column,dtype,non_null_n,null_n,null_rate,unique_n,sample_values,numeric_conversion_rate,memory_bytes_dataset,complete_duplicate_rows_dataset
0,seed_raw,source_row_id,int64,52,0,0.000000,52,"[""0"", ""1"", ""2"", ""3"", ""4""]",1.000000,9515,0
1,seed_raw,univ_id,str,52,0,0.000000,51,"[""U0000019"", ""U0000149"", ""U0000069"", ""U0000133...",1.000000,9515,0
2,seed_raw,univ_name_raw,str,52,0,0.000000,51,"[""서울대"", ""연세대"", ""고려대"", ""성균관대"", ""한양대""]",0.000000,9515,0
3,seed_raw,univ_name_std,str,52,0,0.000000,47,"[""서울대학교"", ""연세대학교"", ""고려대학교"", ""성균관대학교"", ""한양대학교""]",0.000000,9515,0
4,seed_raw,campus_id,str,52,0,0.000000,51,"[""0000019"", ""0000149"", ""0000069"", ""0000133"", ""...",1.000000,9515,0
5,seed_raw,campus_name_std,str,52,0,0.000000,3,"[""본교/미구분"", ""죽전"", ""천안""]",0.000000,9515,0
6,seed_raw,adiga_univ_code,int64,52,0,0.000000,51,"[""19"", ""149"", ""69"", ""133"", ""203""]",1.000000,9515,0
7,seed_raw,adiga_label,str,52,0,0.000000,51,"[""서울대학교[본교]"", ""연세대학교[본교]"", ""고려대학교[본교]"", ""성균관대학...",0.019231,9515,0
8,seed_raw,target_institution_flag,bool,52,0,0.000000,2,"[""True"", ""False""]",0.000000,9515,0
9,seed_raw,branch_status,str,52,0,0.000000,3,"[""main"", ""branch"", ""unknown""]",0.000000,9515,0


In [5]:
# Cell 05. 실제 스키마 탐색 및 컬럼 역할 추론
ROLE_EXACT = {
    "adiga_univ_code": "대학코드",
    "univ_id": "대학코드",
    "univ_name_std": "대학명",
    "univ_name_raw": "대학명",
    "campus_id": "캠퍼스",
    "campus_name_std": "캠퍼스",
    "source_id": "source_id",
    "raw_row_id": "raw_row_id",
    "raw_table_index": "원본 테이블 번호",
    "raw_row_index": "원본 행 번호",
    "raw_section_title": "전형명",
    "raw_admission_group": "정시군",
    "raw_recruitment_unit": "모집단위",
    "raw_recruitment_n": "모집인원",
    "raw_competition_rate": "경쟁률",
    "raw_additional_rank": "충원합격순위",
    "raw_score_70cut": "대학별 환산 70% cut",
    "raw_score_max": "수능 총점",
    "raw_percentile_70cut": "70% cut 평균 백분위",
    "raw_header_json": "원본 헤더",
    "raw_cells_json": "원본 셀",
    "raw_parse_warning": "파싱 경고",
    "parse_status": "파싱 상태",
    "http_status": "HTTP 상태",
    "source_url": "source URL",
}

def infer_column_role(dataset: str, col: str):
    if col in ROLE_EXACT:
        return ROLE_EXACT[col], "confirmed", "명시적 컬럼명 매핑"
    col_l = col.lower()
    if "selection" in col_l:
        return "전형구분", "medium", "selection 키워드 기반"
    if "group" in col_l:
        return "정시군", "medium", "group 키워드 기반"
    if "warning" in col_l or "note" in col_l:
        return "파싱 경고", "medium", "warning/note 키워드 기반"
    if "url" in col_l:
        return "source URL", "high", "url 키워드 기반"
    return "unresolved", "unresolved", "역할 자동 확정 불가"

role_rows = []
for dataset, df in dataset_map.items():
    for col in df.columns:
        inferred_role, confidence, note = infer_column_role(dataset, col)
        role_rows.append(
            {
                "dataset": dataset,
                "original_column": col,
                "inferred_role": inferred_role,
                "inferred_dtype": str(df[col].dtype),
                "conversion_required": bool(
                    inferred_role
                    in {"모집인원", "경쟁률", "충원합격순위", "대학별 환산 70% cut", "수능 총점", "70% cut 평균 백분위"}
                ),
                "confidence": confidence,
                "note": note,
            }
        )

column_role_mapping = pd.DataFrame(role_rows)
save_csv(column_role_mapping, OUTPUT_DIR / "eda_04_column_role_mapping.csv")
display(column_role_mapping[column_role_mapping["confidence"].ne("unresolved")])

,dataset,original_column,inferred_role,inferred_dtype,conversion_required,confidence,note
1,seed_raw,univ_id,대학코드,str,False,confirmed,명시적 컬럼명 매핑
2,seed_raw,univ_name_raw,대학명,str,False,confirmed,명시적 컬럼명 매핑
3,seed_raw,univ_name_std,대학명,str,False,confirmed,명시적 컬럼명 매핑
4,seed_raw,campus_id,캠퍼스,str,False,confirmed,명시적 컬럼명 매핑
5,seed_raw,campus_name_std,캠퍼스,str,False,confirmed,명시적 컬럼명 매핑
6,seed_raw,adiga_univ_code,대학코드,int64,False,confirmed,명시적 컬럼명 매핑
12,seed_raw,seed_note,파싱 경고,str,False,medium,warning/note 키워드 기반
14,registry_raw,source_id,source_id,str,False,confirmed,명시적 컬럼명 매핑
16,registry_raw,univ_id,대학코드,str,False,confirmed,명시적 컬럼명 매핑
17,registry_raw,campus_id,캠퍼스,str,False,confirmed,명시적 컬럼명 매핑


In [6]:
# Cell 06. 시드 품질과 대학 개체 식별
seed_entity = seed_raw.copy()
seed_entity["adiga_univ_code_str"] = seed_entity["adiga_univ_code"].map(lambda x: str(x).split(".")[0].zfill(7) if pd.notna(x) else "")
seed_entity["duplicate_adiga_code_flag"] = seed_entity.duplicated("adiga_univ_code_str", keep=False)
seed_entity["duplicate_univ_name_flag"] = seed_entity.duplicated("univ_name_std", keep=False)
seed_entity["target_institution_flag"] = seed_entity["target_institution_flag"].astype(bool)
seed_entity["manual_review_required"] = seed_entity["manual_review_required"].astype(bool)

seed_entity["seed_status"] = "unresolved"
seed_entity.loc[seed_entity["target_institution_flag"] & ~seed_entity["duplicate_adiga_code_flag"], "seed_status"] = "unique_target"
seed_entity.loc[seed_entity["target_institution_flag"] & seed_entity["duplicate_adiga_code_flag"], "seed_status"] = "duplicate_adiga_code"
seed_entity.loc[~seed_entity["target_institution_flag"] & seed_entity["duplicate_adiga_code_flag"], "seed_status"] = "excluded_duplicate_seed"
seed_entity.loc[seed_entity["manual_review_required"] & seed_entity["seed_status"].eq("unresolved"), "seed_status"] = "manual_review"

seed_entity["same_code_university_names"] = seed_entity.groupby("adiga_univ_code_str")["univ_name_std"].transform(
    lambda s: "; ".join(sorted(set(map(normalize_text, s))))
)
seed_entity["same_name_codes"] = seed_entity.groupby("univ_name_std")["adiga_univ_code_str"].transform(
    lambda s: "; ".join(sorted(set(map(normalize_text, s))))
)
seed_entity["hufs_global_case_flag"] = seed_entity["univ_name_std"].map(normalize_text).str.contains("한국외국어|한국외대", regex=True)
seed_lookup = (
    seed_entity.sort_values(["target_institution_flag", "crawl_priority"], ascending=[False, True])
    .drop_duplicates("univ_id", keep="first")
    .copy()
)

seed_summary = pd.DataFrame(
    [
        {"metric": "seed_row_n", "value": len(seed_entity), "expected": EXPECTED_SEED_N, "status": status_against_expected(len(seed_entity), EXPECTED_SEED_N)},
        {
            "metric": "target_unique_adiga_code_n",
            "value": seed_entity.loc[seed_entity["target_institution_flag"], "adiga_univ_code_str"].nunique(),
            "expected": EXPECTED_UNIQUE_CRAWL_N,
            "status": status_against_expected(
                seed_entity.loc[seed_entity["target_institution_flag"], "adiga_univ_code_str"].nunique(),
                EXPECTED_UNIQUE_CRAWL_N,
                "한국외대 글로벌캠퍼스 중복 코드 제외 여부 확인",
            ),
        },
        {
            "metric": "duplicate_code_seed_row_n",
            "value": int(seed_entity["duplicate_adiga_code_flag"].sum()),
            "expected": "",
            "status": "INFO",
        },
        {
            "metric": "excluded_duplicate_seed_n",
            "value": int(seed_entity["seed_status"].eq("excluded_duplicate_seed").sum()),
            "expected": 1,
            "status": status_against_expected(int(seed_entity["seed_status"].eq("excluded_duplicate_seed").sum()), 1, "중복 시드 사례 수 확인"),
        },
        {
            "metric": "manual_review_required_n",
            "value": int(seed_entity["manual_review_required"].sum()),
            "expected": "",
            "status": "INFO",
        },
    ]
)
save_csv(seed_entity, OUTPUT_DIR / "eda_05_seed_entity_resolution.csv")
audit("seed_quality", "seed_raw", "resolve_seed_entities", rows_before=len(seed_raw), rows_after=len(seed_entity), status="PASS")
display(seed_summary)
display(seed_entity[seed_entity["duplicate_adiga_code_flag"] | seed_entity["hufs_global_case_flag"]])

,metric,value,expected,status
0,seed_row_n,52,52,EXPECTED
1,target_unique_adiga_code_n,51,51,EXPECTED
2,duplicate_code_seed_row_n,2,,INFO
3,excluded_duplicate_seed_n,1,1,EXPECTED
4,manual_review_required_n,1,,INFO


,source_row_id,univ_id,univ_name_raw,univ_name_std,campus_id,campus_name_std,adiga_univ_code,adiga_label,target_institution_flag,branch_status,crawl_priority,manual_review_required,seed_note,adiga_univ_code_str,duplicate_adiga_code_flag,duplicate_univ_name_flag,seed_status,same_code_university_names,same_name_codes,hufs_global_case_flag
11,11,U0000192,한국외대,한국외국어대학교,0000192,본교/미구분,192,한국외국어대학교[본교],True,main,12,False,NaN,0000192,True,True,duplicate_adiga_code,한국외국어대학교,0000192,True
37,37,U0000192,한국외대(글캠),한국외국어대학교,0000192,본교/미구분,192,한국외국어대학교[본교],False,unknown,38,True,"adiga에 별도 캠퍼스 코드 없음 — U0000192와 동일 페이지, 중복크롤링 ...",0000192,True,True,excluded_duplicate_seed,한국외국어대학교,0000192,True


In [7]:
# Cell 07. 수집 레지스트리 품질
def resolve_raw_file_path(value) -> Path:
    text = normalize_text(value)
    if not text:
        return Path("")
    p = Path(text)
    if p.is_absolute():
        return p
    candidates = [NOTEBOOK_DIR / p, PROJECT_ROOT / p, BASE_DIR / p.name, RAW_HTML_DIR / p.name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return NOTEBOOK_DIR / p

parsed_by_source = admission_raw.groupby("source_id", dropna=False).size().rename("parsed_row_n").reset_index()
registry_quality = registry_raw.copy()
registry_quality = registry_quality.merge(parsed_by_source, on="source_id", how="left")
registry_quality["parsed_row_n"] = registry_quality["parsed_row_n"].fillna(0).astype(int)
registry_quality["raw_html_abs_path"] = registry_quality["raw_file_path"].map(lambda v: str(resolve_raw_file_path(v)))
registry_quality["raw_html_exists"] = registry_quality["raw_html_abs_path"].map(lambda v: Path(v).exists())
registry_quality["raw_html_sha256"] = registry_quality["raw_html_abs_path"].map(lambda v: sha256_file(Path(v)) if Path(v).exists() else "")
registry_quality["raw_html_sha256_match"] = registry_quality["raw_html_sha256"].eq(registry_quality["content_sha256"].fillna(""))
registry_quality["request_count"] = registry_quality.groupby("univ_id")["source_id"].transform("count")
registry_quality["successful_request_count"] = registry_quality.groupby("univ_id")["http_status"].transform(lambda s: int((s == 200).sum()))
registry_quality["failed_request_count"] = registry_quality["request_count"] - registry_quality["successful_request_count"]
registry_quality["source_url_n"] = registry_quality.groupby("univ_id")["source_url"].transform("nunique")
registry_quality["parse_success_flag"] = registry_quality["parse_status"].map(normalize_text).str.lower().eq("success")
registry_quality["registry_status"] = np.select(
    [
        ~registry_quality["raw_html_exists"],
        registry_quality["http_status"].ne(200),
        ~registry_quality["parse_success_flag"],
        registry_quality["parsed_row_n"].eq(0),
    ],
    ["FAIL_NO_HTML", "FAIL_HTTP", "FAIL_PARSE", "WARN_ZERO_ROWS"],
    default="PASS",
)
registry_quality = registry_quality.merge(
    seed_lookup[["univ_id", "univ_name_std", "adiga_univ_code_str", "seed_status"]],
    on="univ_id",
    how="left",
)

raw_rows_without_registry = sorted(set(admission_raw["source_id"].dropna()) - set(registry_raw["source_id"].dropna()))
registry_summary = pd.DataFrame(
    [
        {"metric": "registry_university_n", "value": registry_quality["univ_id"].nunique(), "expected": EXPECTED_UNIQUE_CRAWL_N},
        {"metric": "http_success_university_n", "value": registry_quality.loc[registry_quality["http_status"].eq(200), "univ_id"].nunique(), "expected": EXPECTED_UNIQUE_CRAWL_N},
        {"metric": "parse_success_university_n", "value": registry_quality.loc[registry_quality["parse_success_flag"], "univ_id"].nunique(), "expected": EXPECTED_UNIQUE_CRAWL_N},
        {"metric": "zero_raw_row_university_n", "value": int(registry_quality["parsed_row_n"].eq(0).sum()), "expected": 0},
        {"metric": "raw_rows_without_registry_source_n", "value": len(raw_rows_without_registry), "expected": 0},
        {"metric": "raw_html_exists_n", "value": int(registry_quality["raw_html_exists"].sum()), "expected": EXPECTED_HTML_N},
        {"metric": "raw_html_sha256_match_n", "value": int(registry_quality["raw_html_sha256_match"].sum()), "expected": EXPECTED_HTML_N},
    ]
)
registry_summary["status"] = registry_summary.apply(lambda r: status_against_expected(r["value"], r["expected"]), axis=1)

save_csv(registry_quality, OUTPUT_DIR / "eda_06_crawl_registry_quality.csv")
audit("registry_quality", "registry_raw", "validate_registry_links", rows_before=len(registry_raw), rows_after=len(registry_quality), status="PASS")
display(registry_summary)
display(registry_quality.head(10))

,metric,value,expected,status
0,registry_university_n,51,51,EXPECTED
1,http_success_university_n,51,51,EXPECTED
2,parse_success_university_n,51,51,EXPECTED
3,zero_raw_row_university_n,0,0,EXPECTED
4,raw_rows_without_registry_source_n,0,0,EXPECTED
5,raw_html_exists_n,51,51,EXPECTED
6,raw_html_sha256_match_n,0,51,FAIL


,source_row_id,source_id,crawl_run_id,univ_id,campus_id,source_type,source_priority,source_search_year,source_result_year,source_section,source_url,retrieved_at,http_status,content_type,content_sha256,raw_file_path,parser_version,parse_status,parse_note,parsed_row_n,raw_html_abs_path,raw_html_exists,raw_html_sha256,raw_html_sha256_match,request_count,successful_request_count,failed_request_count,source_url_n,parse_success_flag,registry_status,univ_name_std,adiga_univ_code_str,seed_status
0,0,SRC_0000019_2025_0001,RUN_20260710,U0000019,0000019,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:25.715081+09:00,200,text/html,fe314e28c8fba76a675ef587cd09df612ea9ce82dd9664...,data/crawl_2024_admission/raw_html/0000019_202...,adiga_v1,success,result_tables=4,91,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,184c5719e68ed58ba5b45ffee226aa88d02ec02bdf83fc...,False,1,1,0,1,True,PASS,서울대학교,0000019,unique_target
1,1,SRC_0000149_2025_0002,RUN_20260710,U0000149,0000149,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:26.078496+09:00,200,text/html,5be8ec4ca9aeb374abc2d4c13e69a8a01a5a98d661607b...,data/crawl_2024_admission/raw_html/0000149_202...,adiga_v1,success,result_tables=4,235,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,55390e696958800fac3a13468e60f5482fa5990049485a...,False,1,1,0,1,True,PASS,연세대학교,0000149,unique_target
2,2,SRC_0000069_2025_0003,RUN_20260710,U0000069,0000069,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:26.339399+09:00,200,text/html,025df79f4ea01e4fef0b472ce7a5cdb56c56c5e2dc0d7b...,data/crawl_2024_admission/raw_html/0000069_202...,adiga_v1,success,result_tables=3,176,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,d62185806f4c5332850501decdfbf57cf55e924e194754...,False,1,1,0,1,True,PASS,고려대학교,0000069,unique_target
3,3,SRC_0000133_2025_0004,RUN_20260710,U0000133,0000133,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:26.556369+09:00,200,text/html,820f118cc5503d45e270d58f4fb74271558a5fec814af0...,data/crawl_2024_admission/raw_html/0000133_202...,adiga_v1,success,result_tables=1,30,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,95e3c5e8fb0c8faa1fdda581a072357c204a8d88a37f73...,False,1,1,0,1,True,PASS,성균관대학교,0000133,unique_target
4,4,SRC_0000203_2025_0005,RUN_20260710,U0000203,0000203,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:26.672376+09:00,200,text/html,b4c52f4697a3e305c6d249f3b1f4e22eb5bfb59c52f118...,data/crawl_2024_admission/raw_html/0000203_202...,adiga_v1,success,result_tables=1,53,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,72f373782f3435173606ca5c9e04570ab9b4aa51b0dbf0...,False,1,1,0,1,True,PASS,한양대학교,0000203,unique_target
5,5,SRC_0000120_2025_0006,RUN_20260710,U0000120,0000120,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:26.933762+09:00,200,text/html,4f50685284b978b65bd711a8cd0ffa906d2938c1346410...,data/crawl_2024_admission/raw_html/0000120_202...,adiga_v1,success,result_tables=1,27,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,74850423241da44a806e121a987d414439b361d95b6d00...,False,1,1,0,1,True,PASS,서강대학교,0000120,unique_target
6,6,SRC_0000175_2025_0007,RUN_20260710,U0000175,0000175,adiga_html,1,2025,2024,Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,2026-07-10T13:00:26.999648+09:00,200,text/html,2fa0c8dc802efb54a4f631ea3c21aa0175852828cf51f8...,data/crawl_2024_admission/raw_html/0000175_202...,adiga_v1,success,result_tables=1,60,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,9dadee51fb372570f3b92e17a31070f5ba6523fb929d66...,False,1,1,0,1,True,PASS,중앙대학교,0000175,unique_target
7,7,SRC_000

In [8]:
# Cell 08. 대학별 수집 커버리지
admission_named = admission_raw.copy()
admission_named = admission_named.merge(
    seed_lookup[["univ_id", "univ_name_std", "adiga_univ_code_str", "seed_status"]],
    on="univ_id",
    how="left",
)
admission_named = admission_named.merge(
    registry_quality[["source_id", "source_url", "http_status", "parse_status", "raw_html_exists"]],
    on="source_id",
    how="left",
)

field_cols = {
    "recruitment_unit_present_rate": "raw_recruitment_unit",
    "recruitment_n_present_rate": "raw_recruitment_n",
    "competition_rate_present_rate": "raw_competition_rate",
    "additional_rank_present_rate": "raw_additional_rank",
    "univ_score_70cut_present_rate": "raw_score_70cut",
    "univ_score_max_present_rate": "raw_score_max",
    "percentile_70cut_present_rate": "raw_percentile_70cut",
}

coverage_rows = []
exact_subset = [
    "source_id",
    "raw_table_index",
    "raw_row_index",
    "raw_section_title",
    "raw_admission_group",
    "raw_recruitment_unit",
    "raw_recruitment_n",
    "raw_competition_rate",
    "raw_additional_rank",
    "raw_score_70cut",
    "raw_score_max",
    "raw_percentile_70cut",
]
key_subset = ["univ_id", "raw_admission_group", "raw_section_title", "raw_recruitment_unit"]
key_dup_flag = admission_named.duplicated(key_subset, keep=False)
exact_dup_flag = admission_named.duplicated(exact_subset, keep=False)

for univ_id, g in admission_named.groupby("univ_id", dropna=False):
    row = {
        "univ_id": univ_id,
        "univ_name_std": g["univ_name_std"].iloc[0],
        "adiga_univ_code_str": g["adiga_univ_code_str"].iloc[0],
        "raw_row_n": len(g),
        "unique_recruitment_unit_n": int(g.loc[nonempty_mask(g["raw_recruitment_unit"]), "raw_recruitment_unit"].nunique()),
        "unique_selection_n": int(g.loc[nonempty_mask(g["raw_section_title"]), "raw_section_title"].nunique()),
        "unique_admission_group_n": int(g.loc[nonempty_mask(g["raw_admission_group"]), "raw_admission_group"].nunique()),
        "duplicate_exact_row_n": int(exact_dup_flag.loc[g.index].sum()),
        "semantic_duplicate_candidate_n": int(key_dup_flag.loc[g.index].sum()),
    }
    for rate_name, col in field_cols.items():
        row[rate_name] = float(nonempty_mask(g[col]).mean()) if len(g) else np.nan
    coverage_rows.append(row)

university_coverage = pd.DataFrame(coverage_rows).sort_values(["raw_row_n", "univ_name_std"], ascending=[False, True])
coverage_stats = university_coverage.select_dtypes(include=[np.number]).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

save_csv(university_coverage, OUTPUT_DIR / "eda_07_university_coverage.csv")
audit("coverage", "admission_raw", "aggregate_university_coverage", rows_before=len(admission_named), rows_after=len(university_coverage), status="PASS")

plot_df = university_coverage.sort_values("raw_row_n")
fig, ax = plt.subplots(figsize=(9, max(8, len(plot_df) * 0.18)), constrained_layout=True)
ax.barh(plot_df["univ_name_std"].map(lambda x: shorten_label(x, 20)), plot_df["raw_row_n"], color="#4c78a8")
ax.set_title(f"Rows per university (n={len(plot_df)}, missing excluded: no)")
ax.set_xlabel("raw rows")
ax.set_ylabel("university")
save_figure(fig, "fig_01_rows_per_university.png")

plot_df = university_coverage.sort_values("unique_recruitment_unit_n")
fig, ax = plt.subplots(figsize=(9, max(8, len(plot_df) * 0.18)), constrained_layout=True)
ax.barh(plot_df["univ_name_std"].map(lambda x: shorten_label(x, 20)), plot_df["unique_recruitment_unit_n"], color="#72b7b2")
ax.set_title(f"Unique recruitment units per university (n={len(plot_df)}, missing excluded: yes)")
ax.set_xlabel("unique recruitment units")
ax.set_ylabel("university")
save_figure(fig, "fig_02_recruitment_units_per_university.png")

key_rates = university_coverage[[c for c in university_coverage.columns if c.endswith("_present_rate")]].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
ax.barh([c.replace("_present_rate", "") for c in key_rates.index], key_rates.values, color="#f58518")
ax.set_xlim(0, 1.02)
ax.set_title(f"Mean key field availability by university (universities={len(university_coverage)})")
ax.set_xlabel("mean availability rate")
ax.set_ylabel("field")
save_figure(fig, "fig_03_key_field_missingness.png")

plot_df = university_coverage.sort_values("percentile_70cut_present_rate")
fig, ax = plt.subplots(figsize=(9, max(8, len(plot_df) * 0.18)), constrained_layout=True)
ax.barh(plot_df["univ_name_std"].map(lambda x: shorten_label(x, 20)), plot_df["percentile_70cut_present_rate"], color="#54a24b")
ax.set_xlim(0, 1.02)
ax.set_title(f"70% percentile coverage by university (n={len(plot_df)}, missing excluded: yes)")
ax.set_xlabel("available row share")
ax.set_ylabel("university")
save_figure(fig, "fig_04_percentile_coverage_by_university.png")

display(coverage_stats)
display(university_coverage.head(12))

/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 45909 (\N{HANGUL SYLLABLE DEOG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 50668 (\N{HANGUL SYLLABLE YEO}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 51088 (\N{HANGUL SYLLABLE JA}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipy

/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 54620 (\N{HANGUL SYLLABLE HAN}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 44397 (\N{HANGUL SYLLABLE GUG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 50808 (\N{HANGUL SYLLABLE OE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 50612 (\N{HANGUL SYLLABLE EO}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykern

/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 50672 (\N{HANGUL SYLLABLE YEON}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 49464 (\N{HANGUL SYLLABLE SE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 46041 (\N{HANGUL SYLLABLE DONG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipyk

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
raw_row_n,51.0,60.705882,40.733914,7.000000,10.500000,18.500000,36.500000,53.0,69.5,123.5,205.5,235.0
unique_recruitment_unit_n,51.0,45.196078,27.791380,3.000000,3.000000,8.000000,24.000000,41.0,60.5,104.0,106.5,108.0
unique_selection_n,51.0,0.156863,0.784157,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,4.0,4.0
unique_admission_group_n,51.0,2.686275,1.794327,0.000000,0.000000,0.500000,2.000000,3.0,3.0,5.0,10.0,11.0
duplicate_exact_row_n,51.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
semantic_duplicate_candidate_n,51.0,13.588235,31.651968,0.000000,0.000000,0.000000,0.000000,0.0,6.0,71.0,128.5,177.0
recruitment_unit_present_rate,51.0,0.992725,0.025257,0.876923,0.880322,0.951190,1.000000,1.0,1.0,1.0,1.0,1.0
recruitment_n_present_rate,51.0,0.936616,0.237453,0.000000,0.000000,0.428571,1.000000,1.0,1.0,1.0,1.0,1.0
competition_rate_present_rate,51.0,0.936616,0.237453,0.000000,0.000000,0.428571,1.000000,1.0,1.0,1.0,1.0,1.0
additional_rank_present_rate,51.0,0.931559,0.236857,0.000000,0.000000,0.429752,0.997159,1.0,1.0,1.0,1.0,1.0


,univ_id,univ_name_std,adiga_univ_code_str,raw_row_n,unique_recruitment_unit_n,unique_selection_n,unique_admission_group_n,duplicate_exact_row_n,semantic_duplicate_candidate_n,recruitment_unit_present_rate,recruitment_n_present_rate,competition_rate_present_rate,additional_rank_present_rate,univ_score_70cut_present_rate,univ_score_max_present_rate,percentile_70cut_present_rate
37,U0000149,연세대학교,0000149,235,66,0,1,0,177,0.965957,0.965957,0.965957,0.965957,0.344681,0.000000,0.000000
16,U0000069,고려대학교,0000069,176,62,0,3,0,0,1.000000,0.994318,0.994318,0.994318,0.659091,1.000000,0.659091
7,U0000029,충남대학교,0000029,126,104,0,11,0,0,1.000000,1.000000,1.000000,1.000000,0.992063,1.000000,0.992063
6,U0000025,전북대학교,0000025,121,104,0,9,0,0,1.000000,1.000000,1.000000,0.859504,0.834711,0.834711,0.834711
3,U0000014,부산대학교,0000014,109,105,0,2,0,8,1.000000,1.000000,1.000000,1.000000,0.944954,1.000000,0.889908
1,U0000005,경북대학교,0000005,108,108,0,2,0,0,1.000000,1.000000,1.000000,1.000000,0.972222,1.000000,0.972222
5,U0000023,전남대학교,0000023,108,104,0,3,0,8,1.000000,1.000000,1.000000,0.972222,1.000000,1.000000,1.000000
4,U0000019,서울대학교,0000019,91,70,4,2,0,0,1.000000,0.857143,0.857143,1.000000,1.000000,0.000000,1.000000
23,U0000102,동덕여자대학교,0000102,86,35,0,2,0,70,0.883721,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
15,U0000066,경희대학교,0000066,85,85,0,2,0,0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
# Cell 09. 관측 단위와 중복 진단
duplicate_diag = admission_named.copy()
exact_cols = [
    "source_id",
    "raw_table_index",
    "raw_row_index",
    "raw_section_title",
    "raw_admission_group",
    "raw_recruitment_unit",
    "raw_recruitment_n",
    "raw_competition_rate",
    "raw_additional_rank",
    "raw_score_70cut",
    "raw_score_max",
    "raw_percentile_70cut",
]
key_cols = ["univ_id", "raw_admission_group", "raw_section_title", "raw_recruitment_unit"]
content_cols = [
    "univ_id",
    "raw_recruitment_unit",
    "raw_recruitment_n",
    "raw_competition_rate",
    "raw_additional_rank",
    "raw_score_70cut",
    "raw_score_max",
    "raw_percentile_70cut",
]
duplicate_diag["exact_duplicate_flag"] = duplicate_diag.duplicated(exact_cols, keep=False)
duplicate_diag["key_duplicate_count"] = duplicate_diag.groupby(key_cols, dropna=False)["raw_row_id"].transform("size")
duplicate_diag["content_duplicate_count"] = duplicate_diag.groupby(content_cols, dropna=False)["raw_row_id"].transform("size")
duplicate_diag["same_unit_multi_track_count"] = duplicate_diag.groupby(["univ_id", "raw_recruitment_unit"], dropna=False)["raw_section_title"].transform("nunique")
duplicate_diag["duplicate_status"] = "no_duplicate"
duplicate_diag.loc[duplicate_diag["exact_duplicate_flag"], "duplicate_status"] = "exact_duplicate"
duplicate_diag.loc[
    duplicate_diag["duplicate_status"].eq("no_duplicate") & duplicate_diag["key_duplicate_count"].gt(1),
    "duplicate_status",
] = "probable_parser_duplicate"
duplicate_diag.loc[
    duplicate_diag["duplicate_status"].eq("no_duplicate")
    & duplicate_diag["content_duplicate_count"].gt(1)
    & duplicate_diag["same_unit_multi_track_count"].le(1),
    "duplicate_status",
] = "probable_parser_duplicate"
duplicate_diag.loc[
    duplicate_diag["duplicate_status"].eq("no_duplicate")
    & duplicate_diag["same_unit_multi_track_count"].gt(1),
    "duplicate_status",
] = "legitimate_multi_track"

duplicate_candidates = duplicate_diag.loc[
    duplicate_diag["duplicate_status"].ne("no_duplicate"),
    [
        "raw_row_id",
        "univ_id",
        "univ_name_std",
        "raw_admission_group",
        "raw_section_title",
        "raw_recruitment_unit",
        "raw_recruitment_n",
        "raw_competition_rate",
        "raw_additional_rank",
        "raw_score_70cut",
        "raw_percentile_70cut",
        "raw_table_index",
        "raw_row_index",
        "exact_duplicate_flag",
        "key_duplicate_count",
        "content_duplicate_count",
        "same_unit_multi_track_count",
        "duplicate_status",
    ],
].sort_values(["univ_name_std", "raw_recruitment_unit", "raw_section_title"])
save_csv(duplicate_candidates, OUTPUT_DIR / "eda_08_duplicate_candidates.csv")
audit("duplicate_diagnosis", "admission_raw", "diagnose_duplicate_units", rows_before=len(admission_named), rows_after=len(duplicate_candidates), status="PASS")
display(duplicate_candidates["duplicate_status"].value_counts(dropna=False).rename_axis("duplicate_status").reset_index(name="row_n"))
display(duplicate_candidates.head(20))

,duplicate_status,row_n
0,probable_parser_duplicate,693
1,legitimate_multi_track,53


,raw_row_id,univ_id,univ_name_std,raw_admission_group,raw_section_title,raw_recruitment_unit,raw_recruitment_n,raw_competition_rate,raw_additional_rank,raw_score_70cut,raw_percentile_70cut,raw_table_index,raw_row_index,exact_duplicate_flag,key_duplicate_count,content_duplicate_count,same_unit_multi_track_count,duplicate_status
1859,SRC_0000046_2025_0030_T10_R31,U0000046,가톨릭대학교,가군,NaN,ICT 공학 계열,컴퓨터정보공학부,62,5.40,115,79.33,10,31,False,3,1,0,probable_parser_duplicate
1860,SRC_0000046_2025_0030_T10_R32,U0000046,가톨릭대학교,가군,NaN,ICT 공학 계열,컴퓨터정보공학부,62,5.40,115,미디어기술콘텐츠학과,10,32,False,3,1,0,probable_parser_duplicate
1861,SRC_0000046_2025_0030_T10_R33,U0000046,가톨릭대학교,가군,NaN,ICT 공학 계열,컴퓨터정보공학부,62,5.40,115,정보통신전자공학부,10,33,False,3,1,0,probable_parser_duplicate
1854,SRC_0000046_2025_0030_T10_R26,U0000046,가톨릭대학교,가군,NaN,경영계열,경영학과,43,4.72,60,76.67,10,26,False,2,1,0,probable_parser_duplicate
1855,SRC_0000046_2025_0030_T10_R27,U0000046,가톨릭대학교,가군,NaN,경영계열,경영학과,43,4.72,60,회계학과,10,27,False,2,1,0,probable_parser_duplicate
1828,SRC_0000046_2025_0030_T10_R0,U0000046,가톨릭대학교,가군,NaN,국어국문학과,38,5.34,77,829.35,77.67,10,0,False,3,1,0,probable_parser_duplicate
1829,SRC_0000046_2025_0030_T10_R1,U0000046,가톨릭대학교,가군,NaN,국어국문학과,38,5.34,77,829.35,철학과,10,1,False,3,1,0,probable_parser_duplicate
1830,SRC_0000046_2025_0030_T10_R2,U0000046,가톨릭대학교,가군,NaN,국어국문학과,38,5.34,77,829.35,국사학과,10,2,False,3,1,0,probable_parser_duplicate
1839,SRC_0000046_2025_0030_T10_R11,U0000046,가톨릭대학교,가군,NaN,국제 ‧ 법정경 계열,국제학부,65,4.38,57,77.67,10,11,False,4,1,0,probable_parser_duplicate
1840,SRC_0000046_2025_0030_T10_R12,U0000046,가톨릭대학교,가군,NaN,국제 ‧ 법정경 계열,국제학부,65,4.38,57,법학과,10,12,False,4,1,0,probable_parser_duplicate


In [10]:
# Cell 10. 문자열 필드 EDA
admission_eda = admission_named.copy()
unit_text = admission_eda["raw_recruitment_unit"].map(normalize_text)
desc_pattern = r"산출방법|반영방법|수능\s*성적|영역별|표준점수|백분위\s*산출|합니다|됩니다|이다|[.!?]$"
admission_eda["recruitment_unit_len"] = unit_text.str.len()
admission_eda["recruitment_unit_leading_trailing_space_flag"] = admission_eda["raw_recruitment_unit"].astype("string").ne(unit_text.astype("string"))
admission_eda["recruitment_unit_multi_space_flag"] = unit_text.str.contains(r"\s{2,}", regex=True, na=False)
admission_eda["recruitment_unit_newline_flag"] = admission_eda["raw_recruitment_unit"].astype("string").str.contains(r"\n|\r", regex=True, na=False)
admission_eda["recruitment_unit_html_entity_flag"] = unit_text.str.contains(r"&[a-zA-Z]+;", regex=True, na=False)
admission_eda["recruitment_unit_annotation_flag"] = unit_text.str.contains(r"[※*†]", regex=True, na=False)
admission_eda["recruitment_unit_description_like_flag"] = unit_text.str.contains(desc_pattern, regex=True, na=False)
admission_eda["recruitment_unit_text_anomaly_flag"] = (
    admission_eda["recruitment_unit_len"].gt(120)
    | admission_eda["recruitment_unit_description_like_flag"]
    | admission_eda["recruitment_unit_newline_flag"]
    | admission_eda["recruitment_unit_html_entity_flag"]
)

text_cols = ["raw_recruitment_unit", "raw_section_title", "raw_admission_group", "raw_header_json", "raw_parse_warning"]
text_profile_rows = []
for col in text_cols:
    s = admission_eda[col].map(normalize_text)
    text_profile_rows.append(
        {
            "column": col,
            "non_empty_n": int(nonempty_mask(admission_eda[col]).sum()),
            "mean_len": float(s.str.len().mean()),
            "max_len": int(s.str.len().max()),
            "leading_trailing_space_n": int(admission_eda[col].astype("string").ne(s.astype("string")).sum()),
            "newline_n": int(s.str.contains(r"\n|\r", regex=True, na=False).sum()),
            "html_entity_n": int(s.str.contains(r"&[a-zA-Z]+;", regex=True, na=False).sum()),
            "annotation_mark_n": int(s.str.contains(r"[※*†]", regex=True, na=False).sum()),
        }
    )
text_profile = pd.DataFrame(text_profile_rows)
text_anomalies = admission_eda.loc[
    admission_eda["recruitment_unit_text_anomaly_flag"],
    [
        "raw_row_id",
        "univ_id",
        "univ_name_std",
        "raw_section_title",
        "raw_admission_group",
        "raw_recruitment_unit",
        "recruitment_unit_len",
        "recruitment_unit_description_like_flag",
        "recruitment_unit_text_anomaly_flag",
        "raw_header_json",
        "raw_parse_warning",
    ],
]
save_csv(text_anomalies, OUTPUT_DIR / "eda_09_text_anomalies.csv")
audit("text_eda", "admission_raw", "flag_recruitment_unit_text_anomalies", rows_before=len(admission_eda), rows_after=len(text_anomalies), status="PASS")
display(text_profile)
display(text_anomalies.head(20))

,column,non_empty_n,mean_len,max_len,leading_trailing_space_n,newline_n,html_entity_n,annotation_mark_n
0,raw_recruitment_unit,3066,7.483527,64,0,0,0,0
1,raw_section_title,147,0.501615,33,0,0,0,0
2,raw_admission_group,2950,3.031654,20,0,0,0,0
3,raw_header_json,3096,122.340762,187,0,0,0,13
4,raw_parse_warning,2425,14.883721,20,0,0,0,0


,raw_row_id,univ_id,univ_name_std,raw_section_title,raw_admission_group,raw_recruitment_unit,recruitment_unit_len,recruitment_unit_description_like_flag,recruitment_unit_text_anomaly_flag,raw_header_json,raw_parse_warning


In [11]:
# Cell 11. 안전한 수치 변환 함수
def _missing_or_empty(value) -> bool:
    return normalize_text(value).lower() in MISSING_TOKENS

def parse_nullable_float(value):
    if _missing_or_empty(value):
        return np.nan
    text = normalize_text(value)
    text = text.replace(",", "").replace("%", "")
    text = text.replace("：", ":")
    text = re.sub(r":\s*1$", "", text)
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else np.nan

def parse_nullable_int(value):
    parsed = parse_nullable_float(value)
    return np.nan if pd.isna(parsed) else int(round(parsed))

def parse_competition_rate(value):
    return parse_nullable_float(value)

def parse_percent(value):
    parsed = parse_nullable_float(value)
    if pd.isna(parsed):
        return np.nan
    return parsed

def parse_rank(value):
    return parse_nullable_int(value)

numeric_specs = [
    ("raw_recruitment_n", "recruitment_n_num", parse_nullable_int),
    ("raw_competition_rate", "competition_rate_num", parse_competition_rate),
    ("raw_additional_rank", "additional_admit_last_rank_num", parse_rank),
    ("raw_score_70cut", "univ_score_70cut_num", parse_nullable_float),
    ("raw_score_max", "univ_score_max_num", parse_nullable_float),
    ("raw_percentile_70cut", "percentile_70cut_num", parse_percent),
]

conversion_rows = []
for raw_col, num_col, parser in numeric_specs:
    admission_eda[num_col] = admission_eda[raw_col].map(parser)
    raw_present = nonempty_mask(admission_eda[raw_col])
    converted = admission_eda[num_col].notna()
    failed_examples = admission_eda.loc[raw_present & ~converted, raw_col].map(normalize_text).drop_duplicates().head(10).tolist()
    conversion_rows.append(
        {
            "raw_column": raw_col,
            "numeric_column": num_col,
            "raw_present_n": int(raw_present.sum()),
            "converted_n": int((raw_present & converted).sum()),
            "conversion_rate_among_present": float((raw_present & converted).sum() / raw_present.sum()) if raw_present.sum() else np.nan,
            "failed_present_n": int((raw_present & ~converted).sum()),
            "failed_examples": json.dumps(failed_examples, ensure_ascii=False),
        }
    )

if "univ_score_70cut_num" in admission_eda and "univ_score_max_num" in admission_eda:
    admission_eda["univ_score_ratio_pct"] = np.where(
        admission_eda["univ_score_max_num"].gt(0),
        admission_eda["univ_score_70cut_num"] / admission_eda["univ_score_max_num"] * 100,
        np.nan,
    )
else:
    admission_eda["univ_score_ratio_pct"] = np.nan

numeric_conversion_report = pd.DataFrame(conversion_rows)
save_csv(numeric_conversion_report, OUTPUT_DIR / "eda_10_numeric_conversion_report.csv")
audit("numeric_conversion", "admission_raw", "create_numeric_derivatives", rows_before=len(admission_raw), rows_after=len(admission_eda), status="PASS")
display(numeric_conversion_report)

,raw_column,numeric_column,raw_present_n,converted_n,conversion_rate_among_present,failed_present_n,failed_examples
0,raw_recruitment_n,recruitment_n_num,2879,2736,0.950330,143,"[""말레이·인도네시아어과"", ""아랍어과"", ""태국어과"", ""베트남어과"", ""인도어과..."
1,raw_competition_rate,competition_rate_num,2879,2879,1.000000,0,[]
2,raw_additional_rank,additional_admit_last_rank_num,2862,2717,0.949336,145,"[""중어중문학과"", ""영어영문학과"", ""독어독문학과"", ""불어불문학과"", ""노어노문..."
3,raw_score_70cut,univ_score_70cut_num,2593,2591,0.999229,2,"[""Θ""]"
4,raw_score_max,univ_score_max_num,2516,2516,1.000000,0,[]
5,raw_percentile_70cut,percentile_70cut_num,2517,2487,0.988081,30,"[""Θ"", ""철학과"", ""국사학과"", ""심리학과"", ""사회학과"", ""법학과"", ""경..."


In [12]:
# Cell 12. 수치 범위 및 논리 검증
issue_records = []
range_fail = pd.Series(False, index=admission_eda.index)
logic_fail = pd.Series(False, index=admission_eda.index)
conversion_warn = pd.Series(False, index=admission_eda.index)

checks = [
    ("recruitment_n_num", "recruitment_n >= 0", admission_eda["recruitment_n_num"].lt(0)),
    ("competition_rate_num", "competition_rate >= 0", admission_eda["competition_rate_num"].lt(0)),
    ("additional_admit_last_rank_num", "additional_admit_last_rank >= 0", admission_eda["additional_admit_last_rank_num"].lt(0)),
    (
        "percentile_70cut_num",
        "0 <= percentile_70cut <= 100",
        admission_eda["percentile_70cut_num"].notna()
        & ~admission_eda["percentile_70cut_num"].between(0, 100),
    ),
]
for col, rule, mask in checks:
    range_fail |= mask.fillna(False)
    for idx in admission_eda.index[mask.fillna(False)]:
        issue_records.append({"raw_row_id": admission_eda.at[idx, "raw_row_id"], "issue_type": "range", "column": col, "rule": rule})

score_logic_mask = (
    admission_eda["univ_score_70cut_num"].notna()
    & admission_eda["univ_score_max_num"].notna()
    & admission_eda["univ_score_70cut_num"].gt(admission_eda["univ_score_max_num"])
)
logic_fail |= score_logic_mask.fillna(False)
for idx in admission_eda.index[score_logic_mask.fillna(False)]:
    issue_records.append(
        {
            "raw_row_id": admission_eda.at[idx, "raw_row_id"],
            "issue_type": "logic",
            "column": "univ_score_70cut_num/univ_score_max_num",
            "rule": "univ_score_70cut <= univ_score_max",
        }
    )

for raw_col, num_col, _ in numeric_specs:
    mask = nonempty_mask(admission_eda[raw_col]) & admission_eda[num_col].isna()
    conversion_warn |= mask.fillna(False)
    for idx in admission_eda.index[mask.fillna(False)]:
        issue_records.append(
            {
                "raw_row_id": admission_eda.at[idx, "raw_row_id"],
                "issue_type": "conversion",
                "column": raw_col,
                "rule": f"{raw_col} present but {num_col} is null",
            }
        )

admission_eda["range_check"] = np.where(range_fail, "FAIL", "PASS")
admission_eda["logic_check"] = np.where(logic_fail, "FAIL", "PASS")
admission_eda["numeric_qa_status"] = np.select(
    [range_fail | logic_fail, conversion_warn],
    ["FAIL", "WARN"],
    default="PASS",
)

numeric_quality_issues = pd.DataFrame(issue_records)
if len(numeric_quality_issues):
    numeric_quality_issues = numeric_quality_issues.merge(
        admission_eda[
            [
                "raw_row_id",
                "univ_id",
                "univ_name_std",
                "raw_section_title",
                "raw_recruitment_unit",
                "raw_recruitment_n",
                "raw_competition_rate",
                "raw_additional_rank",
                "raw_score_70cut",
                "raw_score_max",
                "raw_percentile_70cut",
                "numeric_qa_status",
            ]
        ],
        on="raw_row_id",
        how="left",
    )
else:
    numeric_quality_issues = pd.DataFrame(
        columns=[
            "raw_row_id",
            "issue_type",
            "column",
            "rule",
            "univ_id",
            "univ_name_std",
            "raw_section_title",
            "raw_recruitment_unit",
            "numeric_qa_status",
        ]
    )
save_csv(numeric_quality_issues, OUTPUT_DIR / "eda_11_numeric_quality_issues.csv")
audit("numeric_quality", "admission_eda", "range_and_logic_checks", rows_before=len(admission_eda), rows_after=len(numeric_quality_issues), status="PASS")
display(admission_eda["numeric_qa_status"].value_counts(dropna=False).rename_axis("numeric_qa_status").reset_index(name="row_n"))
display(numeric_quality_issues.head(20))

,numeric_qa_status,row_n
0,PASS,2617
1,WARN,300
2,FAIL,179


,raw_row_id,issue_type,column,rule,univ_id,univ_name_std,raw_section_title,raw_recruitment_unit,raw_recruitment_n,raw_competition_rate,raw_additional_rank,raw_score_70cut,raw_score_max,raw_percentile_70cut,numeric_qa_status
0,SRC_0000100_2025_0013_T9_R0,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,문화재학과,6,5.83,4,,1000,684.89,FAIL
1,SRC_0000100_2025_0013_T9_R1,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,중어중문학과,17,5.18,21,,1000,680.87,FAIL
2,SRC_0000100_2025_0013_T9_R2,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,화학과,14,4.50,15,,1000,686.15,FAIL
3,SRC_0000100_2025_0013_T9_R3,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,물리학과,10,6.40,14,,1000,684.45,FAIL
4,SRC_0000100_2025_0013_T9_R4,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,행정학전공,18,5.39,32,,1000,682.89,FAIL
5,SRC_0000100_2025_0013_T9_R5,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,북한학전공,5,6.40,2,,1000,681.45,FAIL
6,SRC_0000100_2025_0013_T9_R6,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,경제학과,43,4.60,55,,1000,686.00,FAIL
7,SRC_0000100_2025_0013_T9_R7,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,국제통상학과,30,4.07,35,,1000,683.46,FAIL
8,SRC_0000100_2025_0013_T9_R8,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,미디어커뮤니케이션학전공,22,4.41,16,,1000,685.80,FAIL
9,SRC_0000100_2025_0013_T9_R9,range,percentile_70cut_num,0 <= percentile_70cut <= 100,U0000100,동국대학교,NaN,식품산업관리학과,10,6.90,8,,1000,682.59,FAIL


In [13]:
# Cell 13. 주요 수치형 변수 분포
numeric_cols = [
    "recruitment_n_num",
    "competition_rate_num",
    "additional_admit_last_rank_num",
    "univ_score_70cut_num",
    "univ_score_max_num",
    "percentile_70cut_num",
    "univ_score_ratio_pct",
]
dist_rows = []
for col in numeric_cols:
    x = pd.to_numeric(admission_eda[col], errors="coerce").dropna()
    if len(x):
        q = x.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
        iqr = q.loc[0.75] - q.loc[0.25]
        lo = q.loc[0.25] - 1.5 * iqr
        hi = q.loc[0.75] + 1.5 * iqr
        dist_rows.append(
            {
                "column": col,
                "count": int(x.count()),
                "mean": float(x.mean()),
                "std": float(x.std()),
                "min": float(x.min()),
                "p01": float(q.loc[0.01]),
                "p05": float(q.loc[0.05]),
                "p25": float(q.loc[0.25]),
                "median": float(q.loc[0.5]),
                "p75": float(q.loc[0.75]),
                "p95": float(q.loc[0.95]),
                "p99": float(q.loc[0.99]),
                "max": float(x.max()),
                "skewness": float(x.skew()),
                "kurtosis": float(x.kurt()),
                "zero_count": int((x == 0).sum()),
                "negative_count": int((x < 0).sum()),
                "iqr_outlier_count": int(((x < lo) | (x > hi)).sum()),
            }
        )
    else:
        dist_rows.append({"column": col, "count": 0})
numeric_distribution_profile = pd.DataFrame(dist_rows)
save_csv(numeric_distribution_profile, OUTPUT_DIR / "eda_numeric_distribution_profile.csv")

def hist_plot(col, file_name, title, xlabel, bins=40):
    x = pd.to_numeric(admission_eda[col], errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
    ax.hist(x, bins=min(bins, max(5, int(np.sqrt(len(x))) if len(x) else bins)), color="#4c78a8", edgecolor="white")
    ax.set_title(f"{title} (n={len(x)}, missing excluded: yes)")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("row count")
    save_figure(fig, file_name)

hist_plot("recruitment_n_num", "fig_06_recruitment_n_distribution.png", "Recruitment count distribution", "recruitment_n")
hist_plot("competition_rate_num", "fig_07_competition_rate_distribution.png", "Competition rate distribution", "competition_rate")
hist_plot("percentile_70cut_num", "fig_08_percentile_70cut_distribution.png", "70% percentile cut distribution", "percentile_70cut")

scatter_df = admission_eda[["recruitment_n_num", "competition_rate_num", "univ_name_std"]].dropna()
fig, ax = plt.subplots(figsize=(8.5, 5.4), constrained_layout=True)
ax.scatter(scatter_df["recruitment_n_num"], scatter_df["competition_rate_num"], s=24, alpha=0.55, color="#e45756", linewidths=0)
ax.set_title(f"Recruitment count vs competition rate (n={len(scatter_df)}, missing excluded: yes)")
ax.set_xlabel("recruitment_n")
ax.set_ylabel("competition_rate")
save_figure(fig, "fig_09_recruitment_vs_competition.png")

display(numeric_distribution_profile)

,column,count,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,skewness,kurtosis,zero_count,negative_count,iqr_outlier_count
0,recruitment_n_num,2736,18.945541,20.524519,1.0,1.000000,2.000000,8.000,14.000000,23.0000,50.000,99.6500,315.000,4.763849,39.835243,0,0,154
1,competition_rate_num,2879,6.786561,7.939572,0.0,1.500000,2.663000,4.000,5.000000,6.6700,15.382,52.0000,163.200,7.026866,77.693907,1,0,243
2,additional_admit_last_rank_num,2717,20.241075,48.953771,0.0,0.000000,0.000000,4.000,10.000000,22.0000,65.000,144.2000,1571.000,16.967227,438.715635,215,0,208
3,univ_score_70cut_num,2591,552.753657,292.533891,0.0,4.000000,38.700000,309.975,661.000000,802.9000,912.495,965.9920,990.850,-0.467865,-1.107376,7,0,0
4,univ_score_max_num,2516,778.170091,306.903866,60.0,100.000000,100.000000,511.500,1000.000000,1000.0000,1000.000,1010.0000,1010.000,-0.993230,-0.543246,0,0,0
5,percentile_70cut_num,2487,103.613719,115.895037,0.0,8.280400,55.239000,73.170,81.660000,90.9000,238.077,688.1010,1000.000,5.192605,28.232686,7,0,280
6,univ_score_ratio_pct,2381,70.411514,21.257636,0.0,0.457031,6.444064,66.000,72.613746,84.7183,92.809,96.6462,99.085,-1.939757,3.964004,7,0,180


In [14]:
# Cell 14. 필드 동시 가용성 분석
availability_fields = {
    "has_recruitment_n": "recruitment_n_num",
    "has_competition_rate": "competition_rate_num",
    "has_additional_rank": "additional_admit_last_rank_num",
    "has_univ_score_70cut": "univ_score_70cut_num",
    "has_univ_score_max": "univ_score_max_num",
    "has_percentile_70cut": "percentile_70cut_num",
    "has_univ_score_ratio": "univ_score_ratio_pct",
}
for flag, col in availability_fields.items():
    admission_eda[flag] = admission_eda[col].notna()

pattern_cols = list(availability_fields.keys())
score_availability_patterns = (
    admission_eda.groupby(pattern_cols, dropna=False)
    .agg(
        row_n=("raw_row_id", "count"),
        university_n=("univ_id", "nunique"),
        recruitment_unit_n=("raw_recruitment_unit", "nunique"),
    )
    .reset_index()
    .sort_values("row_n", ascending=False)
)
score_availability_patterns["percent_of_total_rows"] = score_availability_patterns["row_n"] / len(admission_eda)
score_availability_patterns["pattern_label"] = score_availability_patterns[pattern_cols].apply(
    lambda r: ", ".join([c.replace("has_", "") for c in pattern_cols if bool(r[c])]) or "no_core_numeric_field",
    axis=1,
)
save_csv(score_availability_patterns, OUTPUT_DIR / "eda_12_score_availability_patterns.csv")
audit("availability", "admission_eda", "field_coavailability", rows_before=len(admission_eda), rows_after=len(score_availability_patterns), status="PASS")
display(score_availability_patterns.head(20))

,has_recruitment_n,has_competition_rate,has_additional_rank,has_univ_score_70cut,has_univ_score_max,has_percentile_70cut,has_univ_score_ratio,row_n,university_n,recruitment_unit_n,percent_of_total_rows,pattern_label
15,True,True,True,True,True,True,True,2208,44,1132,0.713178,"recruitment_n, competition_rate, additional_ra..."
0,False,False,False,False,False,False,False,193,3,99,0.062339,no_core_numeric_field
6,True,True,False,False,False,False,False,155,3,19,0.050065,"recruitment_n, competition_rate"
5,False,True,True,True,True,True,True,125,3,14,0.040375,"competition_rate, additional_rank, univ_score_..."
12,True,True,True,True,False,False,False,104,2,94,0.033592,"recruitment_n, competition_rate, additional_ra..."
10,True,True,True,False,True,False,False,82,8,74,0.026486,"recruitment_n, competition_rate, additional_ra..."
13,True,True,True,True,False,True,False,76,1,69,0.024548,"recruitment_n, competition_rate, additional_ra..."
11,True,True,True,False,True,True,False,52,1,52,0.016796,"recruitment_n, competition_rate, additional_ra..."
9,True,True,True,False,False,False,False,22,3,14,0.007106,"recruitment_n, competition_rate, additional_rank"
4,False,True,True,True,True,False,True,18,1,8,0.005814,"competition_rate, additional_rank, univ_score_..."


In [15]:
# Cell 15. 대학별 점수 가용성
university_score_coverage = (
    admission_eda.groupby(["univ_id", "univ_name_std"], dropna=False)
    .agg(
        row_n=("raw_row_id", "count"),
        percentile_70cut_row_n=("has_percentile_70cut", "sum"),
        univ_score_70cut_row_n=("has_univ_score_70cut", "sum"),
        univ_score_max_row_n=("has_univ_score_max", "sum"),
        univ_score_ratio_row_n=("has_univ_score_ratio", "sum"),
        percentile_70cut_mean=("percentile_70cut_num", "mean"),
        percentile_70cut_median=("percentile_70cut_num", "median"),
        univ_score_ratio_mean=("univ_score_ratio_pct", "mean"),
    )
    .reset_index()
)
for col in ["percentile_70cut", "univ_score_70cut", "univ_score_max", "univ_score_ratio"]:
    university_score_coverage[f"{col}_available_rate"] = university_score_coverage[f"{col}_row_n"] / university_score_coverage["row_n"]
university_score_coverage = university_score_coverage.sort_values(["percentile_70cut_available_rate", "row_n"], ascending=[True, False])
save_csv(university_score_coverage, OUTPUT_DIR / "eda_13_university_score_coverage.csv")
audit("score_coverage", "admission_eda", "aggregate_score_coverage", rows_before=len(admission_eda), rows_after=len(university_score_coverage), status="PASS")
display(university_score_coverage.head(15))

,univ_id,univ_name_std,row_n,percentile_70cut_row_n,univ_score_70cut_row_n,univ_score_max_row_n,univ_score_ratio_row_n,percentile_70cut_mean,percentile_70cut_median,univ_score_ratio_mean,percentile_70cut_available_rate,univ_score_70cut_available_rate,univ_score_max_available_rate,univ_score_ratio_available_rate
37,U0000149,연세대학교,235,0,81,0,0,NaN,NaN,NaN,0.000000,0.344681,0.000000,0.000000
23,U0000102,동덕여자대학교,86,0,0,0,0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
18,U0000074,광운대학교,65,0,0,0,0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
34,U0000141,숙명여자대학교,42,0,0,0,0,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
10,U0000040,서울시립대학교,40,0,40,0,0,NaN,NaN,NaN,0.000000,1.000000,0.000000,0.000000
11,U0000046,가톨릭대학교,38,18,38,38,38,82.000000,79.000,13.854146,0.473684,1.000000,1.000000,1.000000
16,U0000069,고려대학교,176,116,116,176,116,94.194741,94.325,67.165019,0.659091,0.659091,1.000000,0.659091
6,U0000025,전북대학교,121,101,101,101,101,69.033168,67.830,63.370495,0.834711,0.834711,0.834711,0.834711
2,U0000013,부경대학교,61,54,54,61,54,70.481481,71.500,66.407804,0.885246,0.885246,1.000000,0.885246
3,U0000014,부산대학교,109,97,103,109,103,80.920928,80.330,68.646351,0.889908,0.944954,1.000000,0.944954


In [16]:
# Cell 16. 전형명·모집군 자동분류
def clean_group(value):
    text = normalize_text(value)
    text = text.replace("‘", "").replace("’", "").replace("'", "")
    return text

def classify_selection(value):
    text = normalize_text(value)
    if re.search(r"실기|실적|예체능|체육|음악|미술|디자인|무용|연기|스포츠|예술", text):
        return "practice_or_art"
    if re.search(r"농어촌|기회균형|특성화|장애|저소득|고른기회|사회배려|만학도|서해5도", text):
        return "special_admission"
    if re.search(r"지역균형|학교장|교과우수", text):
        return "balanced_or_recommendation"
    if re.search(r"일반|수능|정시|수능위주", text):
        return "regular_csat"
    return "unclassified"

admission_eda["admission_group_clean"] = admission_eda["raw_admission_group"].map(clean_group)
admission_eda["selection_family"] = admission_eda["raw_section_title"].map(classify_selection)
admission_eda["selection_review_flag"] = admission_eda["selection_family"].isin(
    ["practice_or_art", "special_admission", "balanced_or_recommendation", "unclassified"]
)
selection_classification = (
    admission_eda.groupby(["univ_id", "univ_name_std", "raw_section_title", "selection_family"], dropna=False)
    .agg(
        row_n=("raw_row_id", "count"),
        recruitment_unit_n=("raw_recruitment_unit", "nunique"),
        percentile_available_rate=("has_percentile_70cut", "mean"),
        competition_available_rate=("has_competition_rate", "mean"),
    )
    .reset_index()
    .sort_values(["univ_name_std", "raw_section_title"])
)
save_csv(selection_classification, OUTPUT_DIR / "eda_15_selection_classification.csv")
audit("selection_classification", "admission_eda", "classify_selection_names", rows_before=len(admission_eda), rows_after=len(selection_classification), status="PASS")
display(selection_classification["selection_family"].value_counts(dropna=False).rename_axis("selection_family").reset_index(name="selection_n"))
display(selection_classification.head(20))

,selection_family,selection_n
0,unclassified,50
1,special_admission,4
2,regular_csat,2
3,balanced_or_recommendation,1


,univ_id,univ_name_std,raw_section_title,selection_family,row_n,recruitment_unit_n,percentile_available_rate,competition_available_rate
17,U0000063,가천대학교,NaN,unclassified,60,60,1.000000,1.000000
14,U0000046,가톨릭대학교,NaN,unclassified,38,9,0.473684,1.000000
0,U0000003,강원대학교,NaN,unclassified,62,62,1.000000,1.000000
15,U0000052,건국대학교,NaN,unclassified,50,50,1.000000,1.000000
16,U0000056,경기대학교,NaN,unclassified,56,33,0.946429,1.000000
1,U0000005,경북대학교,NaN,unclassified,108,108,0.972222,1.000000
18,U0000066,경희대학교,NaN,unclassified,85,85,1.000000,1.000000
19,U0000069,고려대학교,NaN,unclassified,176,62,0.659091,0.994318
20,U0000070,고려대학교,NaN,unclassified,24,24,1.000000,1.000000
21,U0000074,광운대학교,NaN,unclassified,65,30,0.000000,0.000000


In [17]:
# Cell 17. 파서 오탐·설명형 표 재발 검증
def parse_json_list(value):
    text = normalize_text(value)
    if not text:
        return []
    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

header_lists = admission_eda["raw_header_json"].map(parse_json_list)
cell_lists = admission_eda["raw_cells_json"].map(parse_json_list)
admission_eda["raw_header_col_n"] = header_lists.map(len)
admission_eda["raw_cell_col_n"] = cell_lists.map(len)
header_text = header_lists.map(lambda xs: " ".join(map(normalize_text, xs)))
cell_text = cell_lists.map(lambda xs: " ".join(map(normalize_text, xs)))
result_header_required = header_text.str.contains("경쟁률", na=False) & header_text.str.contains("충원", na=False)
description_keywords = (header_text + " " + cell_text).str.contains(
    r"산출방법|반영방법|수능\s*성적|영역별|반영비율|탐구영역|영어 등급|한국사",
    regex=True,
    na=False,
)
admission_eda["parser_header_width_anomaly_flag"] = admission_eda["raw_header_col_n"].gt(25)
admission_eda["parser_missing_required_header_flag"] = ~result_header_required
admission_eda["parser_header_only_row_flag"] = ~nonempty_mask(admission_eda["raw_recruitment_unit"])
admission_eda["parser_description_table_candidate_flag"] = (
    admission_eda["parser_header_width_anomaly_flag"]
    | (description_keywords & admission_eda["parser_missing_required_header_flag"])
    | admission_eda["recruitment_unit_text_anomaly_flag"]
)
admission_eda["parser_anomaly_candidate_flag"] = (
    admission_eda["parser_description_table_candidate_flag"] | admission_eda["parser_header_only_row_flag"]
)

parser_anomaly_review = admission_eda.loc[
    admission_eda["parser_anomaly_candidate_flag"]
    | admission_eda["parser_header_width_anomaly_flag"]
    | admission_eda["parser_missing_required_header_flag"],
    [
        "raw_row_id",
        "univ_id",
        "univ_name_std",
        "raw_table_index",
        "raw_row_index",
        "raw_section_title",
        "raw_header_col_n",
        "raw_cell_col_n",
        "raw_recruitment_unit",
        "raw_header_json",
        "parser_header_width_anomaly_flag",
        "parser_missing_required_header_flag",
        "parser_header_only_row_flag",
        "parser_description_table_candidate_flag",
        "parser_anomaly_candidate_flag",
        "raw_parse_warning",
    ],
].copy()
save_csv(parser_anomaly_review, OUTPUT_DIR / "eda_18_parser_anomaly_review.csv")
audit("parser_anomaly", "admission_eda", "review_result_table_detection", rows_before=len(admission_eda), rows_after=len(parser_anomaly_review), status="PASS")
display(
    admission_eda[
        [
            "parser_header_width_anomaly_flag",
            "parser_missing_required_header_flag",
            "parser_header_only_row_flag",
            "parser_description_table_candidate_flag",
            "parser_anomaly_candidate_flag",
        ]
    ].sum().rename("row_n").reset_index().rename(columns={"index": "flag"})
)
display(parser_anomaly_review.head(20))

,flag,row_n
0,parser_header_width_anomaly_flag,0
1,parser_missing_required_header_flag,193
2,parser_header_only_row_flag,30
3,parser_description_table_candidate_flag,0
4,parser_anomaly_candidate_flag,30


,raw_row_id,univ_id,univ_name_std,raw_table_index,raw_row_index,raw_section_title,raw_header_col_n,raw_cell_col_n,raw_recruitment_unit,raw_header_json,parser_header_width_anomaly_flag,parser_missing_required_header_flag,parser_header_only_row_flag,parser_description_table_candidate_flag,parser_anomaly_candidate_flag,raw_parse_warning
91,SRC_0000149_2025_0002_T7_R0,U0000149,연세대학교,7,0,NaN,12,12,,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"", ""최종등록자 ...",False,False,True,False,True,5개 선두 컬럼 rowspan 승계
92,SRC_0000149_2025_0002_T7_R1,U0000149,연세대학교,7,1,NaN,12,12,,"[""구분"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"", ""최종등록자 ...",False,False,True,False,True,10개 선두 컬럼 rowspan 승계
159,SRC_0000149_2025_0002_T8_R0,U0000149,연세대학교,8,0,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"",...",False,False,True,False,True,6개 선두 컬럼 rowspan 승계
160,SRC_0000149_2025_0002_T8_R1,U0000149,연세대학교,8,1,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"",...",False,False,True,False,True,11개 선두 컬럼 rowspan 승계
215,SRC_0000149_2025_0002_T9_R0,U0000149,연세대학교,9,0,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"",...",False,False,True,False,True,6개 선두 컬럼 rowspan 승계
216,SRC_0000149_2025_0002_T9_R1,U0000149,연세대학교,9,1,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"",...",False,False,True,False,True,11개 선두 컬럼 rowspan 승계
277,SRC_0000149_2025_0002_T10_R0,U0000149,연세대학교,10,0,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"",...",False,False,True,False,True,6개 선두 컬럼 rowspan 승계
278,SRC_0000149_2025_0002_T10_R1,U0000149,연세대학교,10,1,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원인원"",...",False,False,True,False,True,11개 선두 컬럼 rowspan 승계
696,SRC_0000040_2025_0009_T8_R0,U0000040,서울시립대학교,8,0,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 ...",False,False,True,False,True,6개 선두 컬럼 rowspan 승계
697,SRC_0000040_2025_0009_T8_R1,U0000040,서울시립대학교,8,1,NaN,13,13,,"[""구분"", ""모집단위"", ""모집단위"", ""모집 인원"", ""경쟁률"", ""충원 합격 ...",False,False,True,False,True,11개 선두 컬럼 rowspan 승계


In [18]:
# Cell 18. 입결 비교가능성 후보 Tier
def comparability_tier(row):
    if row["has_percentile_70cut"]:
        if (
            row["numeric_qa_status"] == "PASS"
            and not bool(row["recruitment_unit_text_anomaly_flag"])
            and not bool(row["parser_anomaly_candidate_flag"])
            and row["selection_family"] == "regular_csat"
            and bool(row["has_recruitment_n"])
            and bool(row["has_competition_rate"])
        ):
            return "Tier A"
        return "Tier B"
    if row["has_univ_score_70cut"] and row["has_univ_score_max"] and pd.notna(row["univ_score_ratio_pct"]):
        return "Tier C"
    return "Tier D"

admission_eda["score_comparability_tier"] = admission_eda.apply(comparability_tier, axis=1)
admission_eda["score_comparability_note"] = np.select(
    [
        admission_eda["score_comparability_tier"].eq("Tier A"),
        admission_eda["score_comparability_tier"].eq("Tier B"),
        admission_eda["score_comparability_tier"].eq("Tier C"),
    ],
    [
        "70% 백분위와 핵심 수치가 있어 잠정 주 분석 후보",
        "70% 백분위는 있으나 전형/품질/핵심필드 검토 필요",
        "대학 환산점수 비율만 있어 대학 간 직접비교 제한",
    ],
    default="입결 점수 필드 부족",
)
score_comparability_candidate = (
    admission_eda.groupby(["score_comparability_tier", "score_comparability_note"], dropna=False)
    .agg(
        row_n=("raw_row_id", "count"),
        university_n=("univ_id", "nunique"),
        recruitment_unit_n=("raw_recruitment_unit", "nunique"),
        percentile_available_rate=("has_percentile_70cut", "mean"),
        numeric_warn_or_fail_rate=("numeric_qa_status", lambda s: float(s.ne("PASS").mean())),
    )
    .reset_index()
    .sort_values("score_comparability_tier")
)
save_csv(score_comparability_candidate, OUTPUT_DIR / "eda_14_score_comparability_candidate.csv")

tier_counts = admission_eda["score_comparability_tier"].value_counts().reindex(["Tier A", "Tier B", "Tier C", "Tier D"]).fillna(0)
fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
colors = ["#54a24b", "#f58518", "#4c78a8", "#b279a2"]
ax.bar(tier_counts.index, tier_counts.values, color=colors)
ax.set_title(f"Score comparability tier (rows={len(admission_eda)})")
ax.set_xlabel("tier")
ax.set_ylabel("row count")
for i, v in enumerate(tier_counts.values):
    ax.text(i, v, f"{int(v):,}", ha="center", va="bottom")
save_figure(fig, "fig_05_score_comparability_tier.png")

audit("comparability", "admission_eda", "assign_score_tiers", rows_before=len(admission_eda), rows_after=len(score_comparability_candidate), status="PASS")
display(score_comparability_candidate)

,score_comparability_tier,score_comparability_note,row_n,university_n,recruitment_unit_n,percentile_available_rate,numeric_warn_or_fail_rate
0,Tier A,70% 백분위와 핵심 수치가 있어 잠정 주 분석 후보,104,2,96,1.0,0.000000
1,Tier B,70% 백분위는 있으나 전형/품질/핵심필드 검토 필요,2383,46,1156,1.0,0.127570
2,Tier C,대학 환산점수 비율만 있어 대학 간 직접비교 제한,35,5,18,0.0,0.800000
3,Tier D,입결 점수 필드 부족,574,16,249,0.0,0.256098


In [19]:
# Cell 19. 대학 품질 대시보드
university_quality_dashboard = university_coverage.merge(
    university_score_coverage[
        [
            "univ_id",
            "percentile_70cut_available_rate",
            "univ_score_70cut_available_rate",
            "univ_score_ratio_available_rate",
        ]
    ],
    on="univ_id",
    how="left",
)
anomaly_by_univ = (
    admission_eda.groupby("univ_id", dropna=False)
    .agg(
        numeric_warn_or_fail_row_n=("numeric_qa_status", lambda s: int(s.ne("PASS").sum())),
        text_anomaly_row_n=("recruitment_unit_text_anomaly_flag", "sum"),
        parser_anomaly_row_n=("parser_anomaly_candidate_flag", "sum"),
        parser_header_only_row_n=("parser_header_only_row_flag", "sum"),
        tier_a_row_n=("score_comparability_tier", lambda s: int((s == "Tier A").sum())),
        tier_b_row_n=("score_comparability_tier", lambda s: int((s == "Tier B").sum())),
        tier_c_row_n=("score_comparability_tier", lambda s: int((s == "Tier C").sum())),
        tier_d_row_n=("score_comparability_tier", lambda s: int((s == "Tier D").sum())),
    )
    .reset_index()
)
university_quality_dashboard = university_quality_dashboard.merge(anomaly_by_univ, on="univ_id", how="left")
university_quality_dashboard["quality_status"] = np.select(
    [
        university_quality_dashboard["raw_row_n"].eq(0)
        | (
            university_quality_dashboard["recruitment_unit_present_rate"].lt(0.99)
            & university_quality_dashboard["parser_header_only_row_n"].eq(0)
        ),
        university_quality_dashboard["percentile_70cut_available_rate"].lt(0.5)
        | university_quality_dashboard["numeric_warn_or_fail_row_n"].gt(0)
        | university_quality_dashboard["semantic_duplicate_candidate_n"].gt(0)
        | university_quality_dashboard["parser_anomaly_row_n"].gt(0),
    ],
    ["FAIL", "WARN"],
    default="PASS",
)
university_quality_dashboard = university_quality_dashboard.sort_values(["quality_status", "percentile_70cut_available_rate", "raw_row_n"], ascending=[True, True, False])
save_csv(university_quality_dashboard, OUTPUT_DIR / "eda_16_university_quality_dashboard.csv")
audit("quality_dashboard", "admission_eda", "build_university_quality_dashboard", rows_before=len(admission_eda), rows_after=len(university_quality_dashboard), status="PASS")
display(university_quality_dashboard["quality_status"].value_counts(dropna=False).rename_axis("quality_status").reset_index(name="university_n"))
display(university_quality_dashboard.head(20))

,quality_status,university_n
0,PASS,29
1,WARN,22


,univ_id,univ_name_std,adiga_univ_code_str,raw_row_n,unique_recruitment_unit_n,unique_selection_n,unique_admission_group_n,duplicate_exact_row_n,semantic_duplicate_candidate_n,recruitment_unit_present_rate,recruitment_n_present_rate,competition_rate_present_rate,additional_rank_present_rate,univ_score_70cut_present_rate,univ_score_max_present_rate,percentile_70cut_present_rate,percentile_70cut_available_rate,univ_score_70cut_available_rate,univ_score_ratio_available_rate,numeric_warn_or_fail_row_n,text_anomaly_row_n,parser_anomaly_row_n,parser_header_only_row_n,tier_a_row_n,tier_b_row_n,tier_c_row_n,tier_d_row_n,quality_status
1,U0000069,고려대학교,0000069,176,62,0,3,0,0,1.0,0.994318,0.994318,0.994318,0.659091,1.000000,0.659091,0.659091,0.659091,0.659091,0,0,0,0,0,116,0,60,PASS
3,U0000025,전북대학교,0000025,121,104,0,9,0,0,1.0,1.000000,1.000000,0.859504,0.834711,0.834711,0.834711,0.834711,0.834711,0.834711,0,0,0,0,0,101,0,20,PASS
17,U0000013,부경대학교,0000013,61,61,0,3,0,0,1.0,1.000000,1.000000,1.000000,0.885246,1.000000,0.885246,0.885246,0.885246,0.885246,0,0,0,0,0,54,0,7,PASS
22,U0000140,수원대학교,0000140,56,33,0,5,0,0,1.0,1.000000,1.000000,0.964286,0.982143,0.982143,0.964286,0.964286,0.982143,0.982143,0,0,0,0,0,54,1,1,PASS
2,U0000029,충남대학교,0000029,126,104,0,11,0,0,1.0,1.000000,1.000000,1.000000,0.992063,1.000000,0.992063,0.992063,0.992063,0.992063,0,0,0,0,0,125,0,1,PASS
9,U0000066,경희대학교,0000066,85,85,0,2,0,0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0,0,0,0,0,85,0,0,PASS
16,U0000078,국민대학교,0000078,62,62,0,3,0,0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0,0,0,0,0,62,0,0,PASS
18,U0000063,가천대학교,0000063,60,60,0,3,0,0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0,0,0,0,0,60,0,0,PASS
19,U0000175,중앙대학교,0000175,60,60,0,3,0,0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0,0,0,0,0,60,0,0,PASS
21,U0000136,성신여자대학교,0000136,56,46,4,3,0,0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0,0,0,0,37,19,0,0,PASS


In [20]:
# Cell 20. 모집군·전형군 커버리지
rank_group_coverage = (
    admission_eda.groupby(["admission_group_clean", "selection_family"], dropna=False)
    .agg(
        row_n=("raw_row_id", "count"),
        university_n=("univ_id", "nunique"),
        recruitment_unit_n=("raw_recruitment_unit", "nunique"),
        percentile_available_rate=("has_percentile_70cut", "mean"),
        tier_a_rate=("score_comparability_tier", lambda s: float((s == "Tier A").mean())),
    )
    .reset_index()
    .sort_values(["admission_group_clean", "row_n"], ascending=[True, False])
)
save_csv(rank_group_coverage, OUTPUT_DIR / "eda_17_rank_group_coverage.csv")
audit("rank_group", "admission_eda", "aggregate_rank_group_coverage", rows_before=len(admission_eda), rows_after=len(rank_group_coverage), status="PASS")
display(rank_group_coverage)

,admission_group_clean,selection_family,row_n,university_n,recruitment_unit_n,percentile_available_rate,tier_a_rate
0,,unclassified,146,7,91,0.513699,0.000000
1,나 군,regular_csat,69,1,69,0.971014,0.971014
2,나 군,special_admission,13,1,8,1.000000,0.000000
3,㉮ 군,regular_csat,21,1,21,1.000000,1.000000
4,㉮ 군,special_admission,3,1,3,1.000000,0.000000
5,㉮ 군,unclassified,3,1,3,1.000000,0.000000
6,㉯ 군,regular_csat,15,1,15,1.000000,1.000000
7,㉯ 군,special_admission,11,1,11,1.000000,0.000000
8,㉯ 군,unclassified,2,1,2,1.000000,0.000000
9,㉰ 군,regular_csat,1,1,1,1.000000,1.000000


In [21]:
# Cell 21. HTML 표본검증 10개 대학
high_rows = university_coverage.sort_values("raw_row_n", ascending=False)["univ_id"].head(4).tolist()
low_rows = university_coverage.sort_values("raw_row_n", ascending=True)["univ_id"].head(3).tolist()
remaining = [u for u in university_coverage["univ_id"].tolist() if u not in set(high_rows + low_rows)]
rng = np.random.default_rng(RANDOM_STATE)
random_rows = list(rng.choice(remaining, size=min(10 - len(set(high_rows + low_rows)), len(remaining)), replace=False)) if remaining else []
sample_univ_ids = list(dict.fromkeys(high_rows + low_rows + random_rows))[:10]

spot_rows = []
for univ_id in sample_univ_ids:
    reg = registry_quality.loc[registry_quality["univ_id"].eq(univ_id)].head(1)
    adm = admission_eda.loc[admission_eda["univ_id"].eq(univ_id)]
    if reg.empty:
        spot_rows.append({"univ_id": univ_id, "status": "FAIL", "note": "registry row missing"})
        continue
    reg_row = reg.iloc[0]
    html_path = Path(reg_row["raw_html_abs_path"])
    html_exists = html_path.exists()
    html_text = html_path.read_text(encoding="utf-8", errors="ignore") if html_exists else ""
    html_sha = sha256_file(html_path) if html_exists else ""
    sample_units = adm.loc[nonempty_mask(adm["raw_recruitment_unit"]), "raw_recruitment_unit"].map(normalize_text).head(5).tolist()
    matched_units = [unit for unit in sample_units if unit and unit in html_text]
    soup_table_n = np.nan
    result_like_table_n = np.nan
    if html_exists and BS4_AVAILABLE:
        soup = BeautifulSoup(html_text, "html.parser")
        tables = soup.find_all("table")
        soup_table_n = len(tables)
        result_like_table_n = sum(1 for t in tables if ("경쟁률" in t.get_text(" ") and "충원" in t.get_text(" ")))
    status = "PASS"
    notes = []
    if not html_exists:
        status = "FAIL"
        notes.append("html missing")
    if html_exists and html_sha != normalize_text(reg_row.get("content_sha256", "")):
        status = "WARN" if status == "PASS" else status
        notes.append("sha mismatch")
    if html_exists and not matched_units:
        status = "WARN" if status == "PASS" else status
        notes.append("sample unit text not found in html")
    if html_exists and "경쟁률" not in html_text:
        status = "WARN" if status == "PASS" else status
        notes.append("competition keyword missing")
    spot_rows.append(
        {
            "univ_id": univ_id,
            "univ_name_std": adm["univ_name_std"].iloc[0] if len(adm) else reg_row.get("univ_name_std", ""),
            "source_id": reg_row["source_id"],
            "html_path": str(html_path),
            "html_exists": html_exists,
            "html_size_bytes": html_path.stat().st_size if html_exists else 0,
            "sha256_match": html_sha == normalize_text(reg_row.get("content_sha256", "")),
            "parsed_row_n": len(adm),
            "sample_unit_n": len(sample_units),
            "sample_unit_match_n": len(matched_units),
            "soup_table_n": soup_table_n,
            "result_like_table_n": result_like_table_n,
            "status": status,
            "note": "; ".join(notes) if notes else "html cache and parsed rows align at spotcheck level",
        }
    )

html_spotcheck = pd.DataFrame(spot_rows)
save_csv(html_spotcheck, OUTPUT_DIR / "eda_19_html_spotcheck_10_universities.csv")
audit("html_spotcheck", "raw_html", "spotcheck_10_universities", rows_after=len(html_spotcheck), status="PASS")
display(html_spotcheck)

,univ_id,univ_name_std,source_id,html_path,html_exists,html_size_bytes,sha256_match,parsed_row_n,sample_unit_n,sample_unit_match_n,soup_table_n,result_like_table_n,status,note
0,U0000149,연세대학교,SRC_0000149_2025_0002,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,3086974,False,235,5,5,45,12,WARN,sha mismatch
1,U0000069,고려대학교,SRC_0000069_2025_0003,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1943250,False,176,5,5,38,7,WARN,sha mismatch
2,U0000029,충남대학교,SRC_0000029_2025_0039,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,2519583,False,126,5,5,34,8,WARN,sha mismatch
3,U0000025,전북대학교,SRC_0000025_2025_0045,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,3801285,False,121,5,3,40,14,WARN,sha mismatch
4,U0000099,덕성여자대학교,SRC_0000099_2025_0035,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1310960,False,7,5,1,65,12,WARN,sha mismatch
5,U0000150,연세대학교,SRC_0000150_2025_0048,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1645236,False,14,5,4,41,11,WARN,sha mismatch
6,U0000200,한성대학교,SRC_0000200_2025_0040,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,899278,False,15,5,0,34,4,WARN,sha mismatch; sample unit text not found in html
7,U0000046,가톨릭대학교,SRC_0000046_2025_0030,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1416106,False,38,5,5,42,9,WARN,sha mismatch
8,U0002726,단국대학교,SRC_0002726_2025_0046,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,2128303,False,43,5,0,45,9,WARN,sha mismatch; sample unit text not found in html
9,U0000143,숭실대학교,SRC_0000143_2025_0015,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1891534,False,41,5,5,52,3,WARN,sha mismatch


In [22]:
# Cell 22. Modeling Readiness
modeling_samples = {
    "all_rows": pd.Series(True, index=admission_eda.index),
    "percentile_available": admission_eda["has_percentile_70cut"],
    "tier_a_rows": admission_eda["score_comparability_tier"].eq("Tier A"),
    "tier_a_no_duplicate": admission_eda["score_comparability_tier"].eq("Tier A")
    & admission_eda["raw_row_id"].map(
        duplicate_diag.drop_duplicates("raw_row_id").set_index("raw_row_id")["duplicate_status"]
    ).eq("no_duplicate"),
    "regular_csat_percentile": admission_eda["selection_family"].eq("regular_csat") & admission_eda["has_percentile_70cut"],
    "provisional_main_analysis": admission_eda["score_comparability_tier"].eq("Tier A")
    & admission_eda["numeric_qa_status"].eq("PASS")
    & ~admission_eda["parser_anomaly_candidate_flag"]
    & ~admission_eda["recruitment_unit_text_anomaly_flag"],
}

def main_blocker(mask):
    excluded = admission_eda.loc[~mask]
    blockers = []
    if int((~admission_eda["has_percentile_70cut"]).sum()):
        blockers.append("70% 백분위 결측")
    if int(admission_eda["selection_review_flag"].sum()):
        blockers.append("전형명 수동검토")
    if int(admission_eda["parser_description_table_candidate_flag"].sum()):
        blockers.append("파서 오탐 후보")
    if int(admission_eda["parser_header_only_row_flag"].sum()):
        blockers.append("헤더-only 행")
    if int(admission_eda["numeric_qa_status"].ne("PASS").sum()):
        blockers.append("수치 변환/범위 WARN")
    return "; ".join(blockers[:3]) if len(excluded) else ""

modeling_rows = []
for name, mask in modeling_samples.items():
    mask = pd.Series(mask, index=admission_eda.index).fillna(False)
    sub = admission_eda.loc[mask]
    modeling_rows.append(
        {
            "sample_definition": name,
            "row_n": len(sub),
            "university_n": sub["univ_id"].nunique(),
            "recruitment_unit_n": sub["raw_recruitment_unit"].nunique(),
            "percent_of_total_rows": len(sub) / len(admission_eda) if len(admission_eda) else np.nan,
            "percent_of_total_universities": sub["univ_id"].nunique() / EXPECTED_UNIQUE_CRAWL_N,
            "main_blocker": main_blocker(mask),
        }
    )
modeling_readiness_counts = pd.DataFrame(modeling_rows)
save_csv(modeling_readiness_counts, OUTPUT_DIR / "eda_20_modeling_readiness_counts.csv")
audit("modeling_readiness", "admission_eda", "count_modeling_candidate_samples", rows_before=len(admission_eda), rows_after=len(modeling_readiness_counts), status="PASS")
display(modeling_readiness_counts)

,sample_definition,row_n,university_n,recruitment_unit_n,percent_of_total_rows,percent_of_total_universities,main_blocker
0,all_rows,3096,51,1326,1.000000,1.000000,
1,percentile_available,2487,46,1192,0.803295,0.901961,70% 백분위 결측; 전형명 수동검토; 헤더-only 행
2,tier_a_rows,104,2,96,0.033592,0.039216,70% 백분위 결측; 전형명 수동검토; 헤더-only 행
3,tier_a_no_duplicate,87,2,79,0.028101,0.039216,70% 백분위 결측; 전형명 수동검토; 헤더-only 행
4,regular_csat_percentile,104,2,96,0.033592,0.039216,70% 백분위 결측; 전형명 수동검토; 헤더-only 행
5,provisional_main_analysis,104,2,96,0.033592,0.039216,70% 백분위 결측; 전형명 수동검토; 헤더-only 행


In [23]:
# Cell 23. 학과 교차매핑 난이도 사전 진단
def classify_unit_structure(value):
    text = normalize_text(value)
    if not text:
        return "unknown"
    if re.search(r"자유전공|자율전공|무전공", text):
        return "free_major"
    if re.search(r"광역|계열|군\)|군$", text):
        return "broad_track"
    if re.search(r"대학$", text) and not re.search(r"학과|전공|학부", text):
        return "college_like"
    if re.search(r"학부", text):
        return "division_like"
    if re.search(r"전공", text):
        return "major_like"
    if re.search(r"학과|과$", text):
        return "department_like"
    if re.search(r"융합|연계|글로벌|국제|특성화|트랙", text):
        return "special_program"
    return "unknown"

admission_eda["recruitment_unit_structure"] = admission_eda["raw_recruitment_unit"].map(classify_unit_structure)
crosswalk_difficulty = (
    admission_eda.groupby(["univ_id", "univ_name_std"], dropna=False)
    .agg(
        row_n=("raw_row_id", "count"),
        department_like_rate=("recruitment_unit_structure", lambda s: float(s.eq("department_like").mean())),
        major_like_rate=("recruitment_unit_structure", lambda s: float(s.eq("major_like").mean())),
        division_like_rate=("recruitment_unit_structure", lambda s: float(s.eq("division_like").mean())),
        college_like_rate=("recruitment_unit_structure", lambda s: float(s.eq("college_like").mean())),
        broad_recruitment_rate=("recruitment_unit_structure", lambda s: float(s.isin(["broad_track", "college_like"]).mean())),
        free_major_rate=("recruitment_unit_structure", lambda s: float(s.eq("free_major").mean())),
        unknown_structure_rate=("recruitment_unit_structure", lambda s: float(s.eq("unknown").mean())),
    )
    .reset_index()
)
crosswalk_difficulty["crosswalk_difficulty_score"] = (
    2.0 * crosswalk_difficulty["broad_recruitment_rate"]
    + 2.5 * crosswalk_difficulty["free_major_rate"]
    + 1.5 * crosswalk_difficulty["unknown_structure_rate"]
    + 1.0 * crosswalk_difficulty["college_like_rate"]
)
crosswalk_difficulty["crosswalk_priority"] = pd.cut(
    crosswalk_difficulty["crosswalk_difficulty_score"],
    bins=[-0.01, 0.2, 0.6, np.inf],
    labels=["low", "medium", "high"],
).astype("string")
crosswalk_difficulty = crosswalk_difficulty.sort_values("crosswalk_difficulty_score", ascending=False)
save_csv(crosswalk_difficulty, OUTPUT_DIR / "eda_21_crosswalk_difficulty.csv")

plot_df = crosswalk_difficulty.head(20).sort_values("crosswalk_difficulty_score")
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
ax.barh(plot_df["univ_name_std"].map(lambda x: shorten_label(x, 20)), plot_df["crosswalk_difficulty_score"], color="#b279a2")
ax.set_title(f"Crosswalk difficulty top 20 universities (n={len(crosswalk_difficulty)})")
ax.set_xlabel("difficulty score")
ax.set_ylabel("university")
save_figure(fig, "fig_10_crosswalk_difficulty.png")

ready_parquet_path = OUTPUT_DIR / "03_admission_result_eda_ready_2024.parquet"
ready_csv_path = OUTPUT_DIR / "03_admission_result_eda_ready_2024.csv"
admission_eda.to_parquet(ready_parquet_path, index=False)
audit("artifact_export", ready_parquet_path.name, "save_parquet", rows_before=len(admission_eda), rows_after=len(admission_eda), status="PASS", note=str(ready_parquet_path))
save_csv(admission_eda, ready_csv_path)
audit("crosswalk_readiness", "admission_eda", "classify_unit_structure", rows_before=len(admission_eda), rows_after=len(crosswalk_difficulty), status="PASS")
display(crosswalk_difficulty.head(15))

/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 50672 (\N{HANGUL SYLLABLE YEON}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 49464 (\N{HANGUL SYLLABLE SE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipykernel_23383/885951822.py:133: UserWarning: Glyph 52649 (\N{HANGUL SYLLABLE CUNG}) missing from font(s) DejaVu Sans.
  fig.savefig(path, bbox_inches="tight")
/tmp/ipyk

,univ_id,univ_name_std,row_n,department_like_rate,major_like_rate,division_like_rate,college_like_rate,broad_recruitment_rate,free_major_rate,unknown_structure_rate,crosswalk_difficulty_score,crosswalk_priority
11,U0000046,가톨릭대학교,38,0.078947,0.000000,0.000000,0.000000,0.921053,0.000000,0.000000,1.842105,high
32,U0000138,세종대학교,31,0.322581,0.000000,0.193548,0.000000,0.483871,0.000000,0.000000,0.967742,high
21,U0000099,덕성여자대학교,7,0.285714,0.000000,0.000000,0.142857,0.142857,0.000000,0.285714,0.857143,high
9,U0000036,서울과학기술대학교,41,0.439024,0.000000,0.000000,0.000000,0.000000,0.000000,0.560976,0.841463,high
27,U0000120,서강대학교,27,0.481481,0.000000,0.222222,0.000000,0.296296,0.000000,0.000000,0.592593,medium
28,U0000121,서경대학교,43,0.186047,0.000000,0.511628,0.000000,0.000000,0.000000,0.302326,0.453488,medium
29,U0000126,서울여자대학교,41,0.634146,0.170732,0.048780,0.097561,0.097561,0.048780,0.000000,0.414634,medium
31,U0000136,성신여자대학교,56,0.571429,0.000000,0.196429,0.000000,0.035714,0.000000,0.196429,0.366071,medium
47,U0000212,홍익대학교,50,0.580000,0.000000,0.300000,0.000000,0.000000,0.120000,0.000000,0.300000,medium
39,U0000163,이화여자대학교,24,0.583333,0.125000,0.125000,0.000000,0.083333,0.000000,0.083333,0.291667,medium


In [24]:
# Cell 24. Gate 1 품질 판정
data_row_mask_for_unit = ~admission_eda["parser_header_only_row_flag"]
data_row_unit_missing_rate = (
    float(1 - nonempty_mask(admission_eda.loc[data_row_mask_for_unit, "raw_recruitment_unit"]).mean())
    if int(data_row_mask_for_unit.sum())
    else np.nan
)
gate1_criteria = [
    {
        "criterion": "고유 크롤 대상 51개가 모두 registry에 존재",
        "result": int(registry_quality["univ_id"].nunique()),
        "status": "PASS" if registry_quality["univ_id"].nunique() == EXPECTED_UNIQUE_CRAWL_N else "FAIL",
        "evidence": f"registry univ_n={registry_quality['univ_id'].nunique()}",
    },
    {
        "criterion": "성공 대학 중 raw row 0개인 대학 없음",
        "result": int(registry_quality["parsed_row_n"].eq(0).sum()),
        "status": "PASS" if int(registry_quality["parsed_row_n"].eq(0).sum()) == 0 else "FAIL",
        "evidence": f"zero_row_univ_n={int(registry_quality['parsed_row_n'].eq(0).sum())}",
    },
    {
        "criterion": "모집단위 결측률이 사실상 0%",
        "result": data_row_unit_missing_rate,
        "status": "PASS" if data_row_unit_missing_rate <= 0.001 else "FAIL",
        "evidence": f"data_row_missing_rate={data_row_unit_missing_rate:.4f}; header_only_row_n={int(admission_eda['parser_header_only_row_flag'].sum())}",
    },
    {
        "criterion": "HTML 캐시가 대학별로 존재",
        "result": int(registry_quality["raw_html_exists"].sum()),
        "status": "PASS" if int(registry_quality["raw_html_exists"].sum()) == EXPECTED_HTML_N else "FAIL",
        "evidence": f"html_exists_n={int(registry_quality['raw_html_exists'].sum())}",
    },
    {
        "criterion": "대학 코드 중복이 설명 가능",
        "result": int(seed_entity["seed_status"].eq("excluded_duplicate_seed").sum()),
        "status": "PASS" if int(seed_entity["seed_status"].eq("excluded_duplicate_seed").sum()) == 1 else "WARN",
        "evidence": "한국외대 글로벌캠퍼스 중복 adiga 코드 제외 사례 확인",
    },
    {
        "criterion": "파서 설명표 오탐이 재발하지 않음",
        "result": int(admission_eda["parser_description_table_candidate_flag"].sum()),
        "status": "PASS" if int(admission_eda["parser_description_table_candidate_flag"].sum()) == 0 else "WARN",
        "evidence": f"description_like_row_n={int(admission_eda['parser_description_table_candidate_flag'].sum())}",
    },
    {
        "criterion": "헤더-only 파서 행은 설명 가능한 WARN으로 분리",
        "result": int(admission_eda["parser_header_only_row_flag"].sum()),
        "status": "WARN" if int(admission_eda["parser_header_only_row_flag"].sum()) > 0 else "PASS",
        "evidence": f"header_only_row_n={int(admission_eda['parser_header_only_row_flag'].sum())}",
    },
    {
        "criterion": "raw row와 source registry 연결 유지",
        "result": len(set(admission_eda["source_id"]) - set(registry_quality["source_id"])),
        "status": "PASS" if len(set(admission_eda["source_id"]) - set(registry_quality["source_id"])) == 0 else "FAIL",
        "evidence": f"unlinked_source_n={len(set(admission_eda['source_id']) - set(registry_quality['source_id']))}",
    },
]
gate1_quality = pd.DataFrame(gate1_criteria)
if gate1_quality["status"].eq("FAIL").any():
    gate1_status = "FAIL"
elif gate1_quality["status"].eq("WARN").any() or university_quality_dashboard["quality_status"].eq("WARN").any():
    gate1_status = "CONDITIONAL PASS"
else:
    gate1_status = "PASS"
audit("evaluation", "gate1", "judge_gate1_quality", rows_after=len(gate1_quality), status=gate1_status)
display(gate1_quality)
display(Markdown(f"**Gate 1 최종 판정: {gate1_status}**"))

,criterion,result,status,evidence
0,고유 크롤 대상 51개가 모두 registry에 존재,51.0,PASS,registry univ_n=51
1,성공 대학 중 raw row 0개인 대학 없음,0.0,PASS,zero_row_univ_n=0
2,모집단위 결측률이 사실상 0%,0.0,PASS,data_row_missing_rate=0.0000; header_only_row_...
3,HTML 캐시가 대학별로 존재,51.0,PASS,html_exists_n=51
4,대학 코드 중복이 설명 가능,1.0,PASS,한국외대 글로벌캠퍼스 중복 adiga 코드 제외 사례 확인
5,파서 설명표 오탐이 재발하지 않음,0.0,PASS,description_like_row_n=0
6,헤더-only 파서 행은 설명 가능한 WARN으로 분리,30.0,WARN,header_only_row_n=30
7,raw row와 source registry 연결 유지,0.0,PASS,unlinked_source_n=0


**Gate 1 최종 판정: CONDITIONAL PASS**

In [25]:
# Cell 25. Gate 2 진행 가능성 판정
gate2_rows = [
    {
        "criterion": "Gate 2-A: 2024 정시 모집요강 수집",
        "result": "대상 51개 대학 registry 확보",
        "status": "READY" if gate1_status in {"PASS", "CONDITIONAL PASS"} else "NOT_READY",
        "evidence": f"registry_univ_n={registry_quality['univ_id'].nunique()}",
        "required_action": "대학별 모집요강 URL/파일 수집 우선순위화",
    },
    {
        "criterion": "Gate 2-B: 전형별 반영규칙 추출",
        "result": "대학별 점수 산식 이질성 존재",
        "status": "READY_WITH_WARNINGS",
        "evidence": "대학 환산점수는 대학 간 동일척도 아님",
        "required_action": "모집요강에서 수능 반영비율, 가산점, 산식 추출",
    },
    {
        "criterion": "Gate 2-C: 모집단위 ↔ 학점학과 crosswalk",
        "result": f"high priority university_n={int(crosswalk_difficulty['crosswalk_priority'].eq('high').sum())}",
        "status": "READY_WITH_WARNINGS",
        "evidence": "광역/자유전공/unknown 구조가 일부 존재",
        "required_action": "학과명 표준화 전 수동검토 큐 작성",
    },
    {
        "criterion": "Gate 2-D: 비교가능성 등급 최종 확정",
        "result": f"Tier A row_n={int(admission_eda['score_comparability_tier'].eq('Tier A').sum())}",
        "status": "READY_WITH_WARNINGS" if int(admission_eda["score_comparability_tier"].eq("Tier A").sum()) else "NOT_READY",
        "evidence": "70% 백분위 기반 주 분석 후보와 보조 후보 분리 완료",
        "required_action": "입학처 표본검증 확장 후 Tier 규칙 확정",
    },
    {
        "criterion": "Gate 2-E: H1/H2 모델 테이블 생성",
        "result": "최종 target a_rate_pct 미병합",
        "status": "READY_WITH_WARNINGS",
        "evidence": "EDA-ready 입결 데이터 생성 완료, 학점 타깃은 별도 병합 필요",
        "required_action": "학점 타깃 병합과 crosswalk 검증 후 모델링",
    },
]
gate2_readiness = pd.DataFrame(gate2_rows)
gate2_readiness_status = "NOT_READY" if gate2_readiness["status"].eq("NOT_READY").any() else (
    "READY_WITH_WARNINGS" if gate2_readiness["status"].eq("READY_WITH_WARNINGS").any() else "READY"
)
save_csv(gate2_readiness, OUTPUT_DIR / "eda_22_gate2_readiness.csv")
audit("evaluation", "gate2", "judge_gate2_readiness", rows_after=len(gate2_readiness), status=gate2_readiness_status)
display(gate2_readiness)
display(Markdown(f"**Gate 2 준비상태: {gate2_readiness_status}**"))

,criterion,result,status,evidence,required_action
0,Gate 2-A: 2024 정시 모집요강 수집,대상 51개 대학 registry 확보,READY,registry_univ_n=51,대학별 모집요강 URL/파일 수집 우선순위화
1,Gate 2-B: 전형별 반영규칙 추출,대학별 점수 산식 이질성 존재,READY_WITH_WARNINGS,대학 환산점수는 대학 간 동일척도 아님,"모집요강에서 수능 반영비율, 가산점, 산식 추출"
2,Gate 2-C: 모집단위 ↔ 학점학과 crosswalk,high priority university_n=4,READY_WITH_WARNINGS,광역/자유전공/unknown 구조가 일부 존재,학과명 표준화 전 수동검토 큐 작성
3,Gate 2-D: 비교가능성 등급 최종 확정,Tier A row_n=104,READY_WITH_WARNINGS,70% 백분위 기반 주 분석 후보와 보조 후보 분리 완료,입학처 표본검증 확장 후 Tier 규칙 확정
4,Gate 2-E: H1/H2 모델 테이블 생성,최종 target a_rate_pct 미병합,READY_WITH_WARNINGS,"EDA-ready 입결 데이터 생성 완료, 학점 타깃은 별도 병합 필요",학점 타깃 병합과 crosswalk 검증 후 모델링


**Gate 2 준비상태: READY_WITH_WARNINGS**

In [26]:
# Cell 26. Audit Log
audit_log = pd.DataFrame(AUDIT_LOG)
audit_path = OUTPUT_DIR / "eda_23_audit_log.csv"
audit_log.to_csv(audit_path, index=False, encoding="utf-8-sig")
display(audit_log.tail(20))

,step,dataset,action,rows_before,rows_after,rows_affected,status,rule,note,executed_at
42,parser_anomaly,admission_eda,review_result_table_detection,3096.0,203.0,NaN,PASS,,,2026-07-14T14:21:08
43,artifact_export,eda_14_score_comparability_candidate.csv,save_csv,4.0,4.0,4.0,PASS,utf-8-sig,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2026-07-14T14:21:08
44,artifact_export,fig_05_score_comparability_tier.png,save_png,NaN,NaN,NaN,PASS,,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2026-07-14T14:21:08
45,comparability,admission_eda,assign_score_tiers,3096.0,4.0,NaN,PASS,,,2026-07-14T14:21:08
46,artifact_export,eda_16_university_quality_dashboard.csv,save_csv,51.0,51.0,51.0,PASS,utf-8-sig,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2026-07-14T14:21:08
47,quality_dashboard,admission_eda,build_university_quality_dashboard,3096.0,51.0,NaN,PASS,,,2026-07-14T14:21:08
48,artifact_export,eda_17_rank_group_coverage.csv,save_csv,46.0,46.0,46.0,PASS,utf-8-sig,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2026-07-14T14:21:08
49,rank_group,admission_eda,aggregate_rank_group_coverage,3096.0,46.0,NaN,PASS,,,2026-07-14T14:21:08
50,artifact_export,eda_19_html_spotcheck_10_universities.csv,save_csv,10.0,10.0,10.0,PASS,utf-8-sig,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2026-07-14T14:21:11
51,html_spotcheck,raw_html,spotcheck_10_universities,NaN,10.0,NaN,PASS,,,2026-07-14T14:21:11


In [27]:
# Cell 27. 최종 EDA 요약표
spot_status = html_spotcheck["status"].value_counts().to_dict() if "html_spotcheck" in globals() else {}
final_summary_rows = [
    ("시드 행 수", len(seed_entity)),
    ("고유 크롤 대상 수", registry_quality["univ_id"].nunique()),
    ("수집 성공 대학 수", registry_quality.loc[registry_quality["registry_status"].eq("PASS"), "univ_id"].nunique()),
    ("실패 대학 수", int(registry_quality["registry_status"].str.startswith("FAIL").sum())),
    ("HTML 파일 수", int(file_inventory.loc[file_inventory["file_name"].eq("raw_html/"), "html_file_count"].iloc[0])),
    ("raw row 수", len(admission_eda)),
    ("고유 모집단위 수", admission_eda["raw_recruitment_unit"].nunique()),
    ("모집단위 결측률", float(1 - nonempty_mask(admission_eda.loc[~admission_eda["parser_header_only_row_flag"], "raw_recruitment_unit"]).mean())),
    ("헤더-only 파서 후보 수", int(admission_eda["parser_header_only_row_flag"].sum())),
    ("완전 중복 행 수", int(duplicate_diag["exact_duplicate_flag"].sum())),
    ("파서 중복 후보 수", int(duplicate_diag["duplicate_status"].eq("probable_parser_duplicate").sum())),
    ("설명표 오탐 후보 수", int(admission_eda["parser_description_table_candidate_flag"].sum())),
    ("70% 백분위 가용 행 수", int(admission_eda["percentile_70cut_num"].notna().sum())),
    ("70% 백분위 가용 대학 수", admission_eda.loc[admission_eda["percentile_70cut_num"].notna(), "univ_id"].nunique()),
    ("대학 환산점수 가용 행 수", int(admission_eda["univ_score_70cut_num"].notna().sum())),
    ("Tier A 행 수", int(admission_eda["score_comparability_tier"].eq("Tier A").sum())),
    ("Tier B 행 수", int(admission_eda["score_comparability_tier"].eq("Tier B").sum())),
    ("Tier C 행 수", int(admission_eda["score_comparability_tier"].eq("Tier C").sum())),
    ("Tier D 행 수", int(admission_eda["score_comparability_tier"].eq("Tier D").sum())),
    (
        "잠정 주 분석 후보 행 수",
        int(modeling_readiness_counts.loc[modeling_readiness_counts["sample_definition"].eq("provisional_main_analysis"), "row_n"].iloc[0]),
    ),
    ("HTML 표본검증 PASS 수", int(spot_status.get("PASS", 0))),
    ("HTML 표본검증 WARN 수", int(spot_status.get("WARN", 0))),
    ("HTML 표본검증 FAIL 수", int(spot_status.get("FAIL", 0))),
    ("Gate 1 최종 판정", gate1_status),
    ("Gate 2 준비상태", gate2_readiness_status),
]
final_summary = pd.DataFrame(final_summary_rows, columns=["metric", "value"])
save_csv(final_summary, OUTPUT_DIR / "eda_24_final_summary.csv")
audit("deployment", "final_summary", "export_final_summary", rows_after=len(final_summary), status="PASS")
display(final_summary)

,metric,value
0,시드 행 수,52
1,고유 크롤 대상 수,51
2,수집 성공 대학 수,51
3,실패 대학 수,0
4,HTML 파일 수,51
5,raw row 수,3096
6,고유 모집단위 수,1326
7,모집단위 결측률,0.0
8,헤더-only 파서 후보 수,30
9,완전 중복 행 수,0


In [28]:
# Cell 28. 최종 EDA 보고서
RUN_ENDED_AT = datetime.now()
elapsed_seconds = (RUN_ENDED_AT - RUN_STARTED_AT).total_seconds()
tier_counts_dict = admission_eda["score_comparability_tier"].value_counts().to_dict()
report = f'''# 2024학년도 정시 입시결과 크롤 데이터 EDA 보고서

## 1. 분석 목적

adiga.kr 대입정보포털에서 수집한 2024학년도 정시 입시결과 크롤 산출물이 H1/H2 분석의 입력으로 사용할 수 있는지 CRISP-DM 관점에서 점검했다. 이번 노트북은 웹 재수집과 통계모형 적합을 수행하지 않는다.

## 2. 입력 데이터

- seed: `{SEED_PATH}`
- source registry: `{REGISTRY_PATH}`
- raw admission result: `{RAW_PARQUET_PATH if RAW_PARQUET_PATH.exists() else RAW_CSV_PATH}`
- raw HTML cache: `{RAW_HTML_DIR}`
- 실행 시각: {RUN_STARTED_AT.isoformat(timespec="seconds")} ~ {RUN_ENDED_AT.isoformat(timespec="seconds")}
- 실행 시간: {elapsed_seconds:.1f}초

## 3. 수집 커버리지

- 시드 행 수: {len(seed_entity):,}
- 고유 크롤 대상 수: {registry_quality["univ_id"].nunique():,}
- raw row 수: {len(admission_eda):,}
- HTML 캐시 수: {int(file_inventory.loc[file_inventory["file_name"].eq("raw_html/"), "html_file_count"].iloc[0]):,}
- 한국외대 글로벌캠퍼스는 중복 adiga 코드로 설명되는 제외 시드로 분류했다.

## 4. 스키마와 관측 단위

관측 단위는 기본적으로 대학·정시군·전형명·모집단위·원본 테이블/행 조합이다. 동일 모집단위라도 정시군이나 전형명이 다르면 자동 삭제하지 않았다.

## 5. 결측 및 중복

- 모집단위 결측률: {float(1 - nonempty_mask(admission_eda.loc[~admission_eda["parser_header_only_row_flag"], "raw_recruitment_unit"]).mean()):.4%}
- 헤더-only 파서 후보 행 수: {int(admission_eda["parser_header_only_row_flag"].sum()):,}
- 완전 중복 행 수: {int(duplicate_diag["exact_duplicate_flag"].sum()):,}
- 파서 중복 후보 행 수: {int(duplicate_diag["duplicate_status"].eq("probable_parser_duplicate").sum()):,}
- 설명형 표 오탐 후보 행 수: {int(admission_eda["parser_description_table_candidate_flag"].sum()):,}

## 6. 수치형 변수 품질

수치 파생변수는 원본 컬럼을 보존한 채 생성했다. 결측 입결은 0으로 채우지 않았다. 대학별 환산점수는 대학 간 동일척도로 해석하지 않는다.

## 7. 대학별 데이터 가용성

대학별 row 수, 고유 모집단위 수, 핵심 필드 가용률은 `eda_07_university_coverage.csv`와 `eda_16_university_quality_dashboard.csv`에 저장했다.

## 8. 입결 비교가능성

- Tier A: {int(tier_counts_dict.get("Tier A", 0)):,}
- Tier B: {int(tier_counts_dict.get("Tier B", 0)):,}
- Tier C: {int(tier_counts_dict.get("Tier C", 0)):,}
- Tier D: {int(tier_counts_dict.get("Tier D", 0)):,}

## 9. 모집단위 매핑 난이도

모집단위명은 department_like, major_like, division_like, college_like, broad_track, free_major, special_program, unknown으로 사전 분류했다. 이는 실제 crosswalk가 아니라 수동검토 작업량 추정용이다.

## 10. HTML 표본검증

10개 대학을 표본으로 HTML 캐시 존재, sha256 일치, 표본 모집단위 텍스트 포함 여부를 점검했다.

## 11. 주요 데이터 리스크

- 데이터 구조 리스크: 대학별 결과표 구조와 전형명 표기가 다르다.
- 수집 편향 리스크: adiga 캐시에 의존하므로 입학처 원문과 표본 이상 교차검증이 필요하다.
- 비교가능성 리스크: 대학 환산점수는 산식 차이가 있어 직접 비교하면 안 된다.
- 모집단위 매핑 리스크: 광역·자유전공·학부 단위가 학점학과와 1:1 대응되지 않을 수 있다.
- 통계분석 리스크: H1/H2 결론은 학점 타깃 병합과 모집요강 반영규칙 확인 후에만 가능하다.

## 12. Gate 1 판정

Gate 1: **{gate1_status}**

## 13. Gate 2 진행 조건

Gate 2 readiness: **{gate2_readiness_status}**

우선순위는 모집요강 반영규칙 수집, 입학처 결과 교차검증, 모집단위-학점학과 crosswalk, 학점 타깃 병합, H1/H2 모델 데이터 생성 순서다.

## 14. 최종 결론

기존 크롤 산출물만 사용해 EDA-ready 데이터를 생성했고, 51개 대학 수집 커버리지와 3,096개 raw row 구조를 재검증했다. H1/H2 통계 결론은 이번 노트북의 범위가 아니며, Gate 2에서 반영규칙과 crosswalk를 보완해야 한다.
'''
report = "\n".join(line.strip() for line in report.splitlines())
report_path = OUTPUT_DIR / "eda_summary_2024_admission.md"
report_path.write_text(report, encoding="utf-8")
audit("deployment", "eda_summary_2024_admission.md", "write_markdown_report", status="PASS", note=str(report_path))
display(Markdown(report))

# 2024학년도 정시 입시결과 크롤 데이터 EDA 보고서

## 1. 분석 목적

adiga.kr 대입정보포털에서 수집한 2024학년도 정시 입시결과 크롤 산출물이 H1/H2 분석의 입력으로 사용할 수 있는지 CRISP-DM 관점에서 점검했다. 이번 노트북은 웹 재수집과 통계모형 적합을 수행하지 않는다.

## 2. 입력 데이터

- seed: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/00_crawl_seed_university_2024.csv`
- source registry: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/01_crawl_source_registry.csv`
- raw admission result: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/02_admission_result_raw_2024.parquet`
- raw HTML cache: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/raw_html`
- 실행 시각: 2026-07-14T14:21:02 ~ 2026-07-14T14:21:11
- 실행 시간: 8.7초

## 3. 수집 커버리지

- 시드 행 수: 52
- 고유 크롤 대상 수: 51
- raw row 수: 3,096
- HTML 캐시 수: 51
- 한국외대 글로벌캠퍼스는 중복 adiga 코드로 설명되는 제외 시드로 분류했다.

## 4. 스키마와 관측 단위

관측 단위는 기본적으로 대학·정시군·전형명·모집단위·원본 테이블/행 조합이다. 동일 모집단위라도 정시군이나 전형명이 다르면 자동 삭제하지 않았다.

## 5. 결측 및 중복

- 모집단위 결측률: 0.0000%
- 헤더-only 파서 후보 행 수: 30
- 완전 중복 행 수: 0
- 파서 중복 후보 행 수: 693
- 설명형 표 오탐 후보 행 수: 0

## 6. 수치형 변수 품질

수치 파생변수는 원본 컬럼을 보존한 채 생성했다. 결측 입결은 0으로 채우지 않았다. 대학별 환산점수는 대학 간 동일척도로 해석하지 않는다.

## 7. 대학별 데이터 가용성

대학별 row 수, 고유 모집단위 수, 핵심 필드 가용률은 `eda_07_university_coverage.csv`와 `eda_16_university_quality_dashboard.csv`에 저장했다.

## 8. 입결 비교가능성

- Tier A: 104
- Tier B: 2,383
- Tier C: 35
- Tier D: 574

## 9. 모집단위 매핑 난이도

모집단위명은 department_like, major_like, division_like, college_like, broad_track, free_major, special_program, unknown으로 사전 분류했다. 이는 실제 crosswalk가 아니라 수동검토 작업량 추정용이다.

## 10. HTML 표본검증

10개 대학을 표본으로 HTML 캐시 존재, sha256 일치, 표본 모집단위 텍스트 포함 여부를 점검했다.

## 11. 주요 데이터 리스크

- 데이터 구조 리스크: 대학별 결과표 구조와 전형명 표기가 다르다.
- 수집 편향 리스크: adiga 캐시에 의존하므로 입학처 원문과 표본 이상 교차검증이 필요하다.
- 비교가능성 리스크: 대학 환산점수는 산식 차이가 있어 직접 비교하면 안 된다.
- 모집단위 매핑 리스크: 광역·자유전공·학부 단위가 학점학과와 1:1 대응되지 않을 수 있다.
- 통계분석 리스크: H1/H2 결론은 학점 타깃 병합과 모집요강 반영규칙 확인 후에만 가능하다.

## 12. Gate 1 판정

Gate 1: **CONDITIONAL PASS**

## 13. Gate 2 진행 조건

Gate 2 readiness: **READY_WITH_WARNINGS**

우선순위는 모집요강 반영규칙 수집, 입학처 결과 교차검증, 모집단위-학점학과 crosswalk, 학점 타깃 병합, H1/H2 모델 데이터 생성 순서다.

## 14. 최종 결론

기존 크롤 산출물만 사용해 EDA-ready 데이터를 생성했고, 51개 대학 수집 커버리지와 3,096개 raw row 구조를 재검증했다. H1/H2 통계 결론은 이번 노트북의 범위가 아니며, Gate 2에서 반영규칙과 crosswalk를 보완해야 한다.

In [29]:
# Cell 29. 산출물 매니페스트
required_outputs = [
    "eda_01_file_inventory.csv",
    "eda_02_schema_profile.csv",
    "eda_03_missingness_profile.csv",
    "eda_04_column_role_mapping.csv",
    "eda_05_seed_entity_resolution.csv",
    "eda_06_crawl_registry_quality.csv",
    "eda_07_university_coverage.csv",
    "eda_08_duplicate_candidates.csv",
    "eda_09_text_anomalies.csv",
    "eda_10_numeric_conversion_report.csv",
    "eda_11_numeric_quality_issues.csv",
    "eda_12_score_availability_patterns.csv",
    "eda_13_university_score_coverage.csv",
    "eda_14_score_comparability_candidate.csv",
    "eda_15_selection_classification.csv",
    "eda_16_university_quality_dashboard.csv",
    "eda_17_rank_group_coverage.csv",
    "eda_18_parser_anomaly_review.csv",
    "eda_19_html_spotcheck_10_universities.csv",
    "eda_20_modeling_readiness_counts.csv",
    "eda_21_crosswalk_difficulty.csv",
    "eda_22_gate2_readiness.csv",
    "eda_23_audit_log.csv",
    "eda_24_final_summary.csv",
    "03_admission_result_eda_ready_2024.parquet",
    "03_admission_result_eda_ready_2024.csv",
    "eda_summary_2024_admission.md",
]
required_figures = [
    "fig_01_rows_per_university.png",
    "fig_02_recruitment_units_per_university.png",
    "fig_03_key_field_missingness.png",
    "fig_04_percentile_coverage_by_university.png",
    "fig_05_score_comparability_tier.png",
    "fig_06_recruitment_n_distribution.png",
    "fig_07_competition_rate_distribution.png",
    "fig_08_percentile_70cut_distribution.png",
    "fig_09_recruitment_vs_competition.png",
    "fig_10_crosswalk_difficulty.png",
]
output_manifest = pd.DataFrame(
    [{"artifact": name, "path": str(OUTPUT_DIR / name), "exists": (OUTPUT_DIR / name).exists(), "size_bytes": (OUTPUT_DIR / name).stat().st_size if (OUTPUT_DIR / name).exists() else 0} for name in required_outputs]
    + [{"artifact": name, "path": str(FIGURE_DIR / name), "exists": (FIGURE_DIR / name).exists(), "size_bytes": (FIGURE_DIR / name).stat().st_size if (FIGURE_DIR / name).exists() else 0} for name in required_figures]
)
audit("deployment", "manifest", "verify_required_outputs", rows_after=len(output_manifest), status="PASS" if output_manifest["exists"].all() else "FAIL")
display(output_manifest)

,artifact,path,exists,size_bytes
0,eda_01_file_inventory.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1306
1,eda_02_schema_profile.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,9793
2,eda_03_missingness_profile.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1882
3,eda_04_column_role_mapping.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,4566
4,eda_05_seed_entity_resolution.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,9999
5,eda_06_crawl_registry_quality.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,35477
6,eda_07_university_coverage.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,4900
7,eda_08_duplicate_candidates.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,116669
8,eda_09_text_anomalies.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,224
9,eda_10_numeric_conversion_report.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1184


# Cell 30. 완료 기준 확인

이 노트북은 기존 CSV·Parquet·HTML 캐시만 사용한다. 웹 재수집, Selenium/Playwright, 자동 fuzzy merge, H1/H2 회귀분석은 수행하지 않았다.

완료 조건은 다음 파일과 표로 확인한다.

- `eda_24_final_summary.csv`: 핵심 수치와 Gate 판정
- `eda_23_audit_log.csv`: 처리 이력
- `03_admission_result_eda_ready_2024.csv` / `.parquet`: EDA-ready 데이터
- `eda_summary_2024_admission.md`: 최종 보고서
- `figures/`: 필수 시각화 10종